# 00 — Dynamique latente structurée : ce qu'un prior spectral achète, et ce qu'il ne peut pas acheter

*Notebook de cours du dépôt `structured-latent-dynamics`.

---

## 0. Ce que ce notebook démontre

La question du dépôt, est une question
**conditionnelle** :

> *Quand une structure dynamique explicite dans le predictor latent d'un world model JEPA
> améliore-t-elle la stabilité de rollout, la suffisance d'état et la planifiabilité — et à
> quel coût par décision réussie ?*

JEPA (Joint-Embedding Predictive Architecture) : un encodeur produit un état latent $z$, un
predictor avance $z$ dans le temps sans jamais redécoder en observations ; c'est ce predictor
latent que ce notebook étudie.

Le mot important est **quand**. Ce notebook n'essaie pas de montrer qu'un prior structurel est
bon. Il essaie de délimiter, aussi précisément que possible, la frontière entre les régimes où
un tel prior peut payer et ceux où il ne le peut pas — et de séparer, à l'intérieur de cette
frontière, ce qui est **démontré**, ce qui est **mesuré**, et ce qui reste **supposé**.

### 0.1 Les trois résultats mesurés qui organisent la lecture

Trois faits, tirés du dépôt, organisent tout ce qui suit. Les deux premiers sont démontrés
ailleurs dans la série ; ce notebook les cite comme motivation et y revient au fil du texte, à
l'endroit précis où leur système conducteur apparaît.

1. **Exprimable n'est pas atteignable.** Sur un système linéaire dont la solution exacte
   appartient à la classe d'hypothèses du modèle *et* se récupère en forme close par moindres
   carrés, le régime d'entraînement déclaré atterrit environ deux décades au-dessus du seuil
   d'exactitude pré-enregistré (seuils fixés et commités avant les runs). Quatre explications
   concurrentes ont été testées et éliminées, chacune par sa propre mesure. Démontré au
   notebook 04 (`04_ce_que_la_grille_a_dit.ipynb`), sur le système conducteur introduit ici en
   §3.
2. **Une dégénérescence de jauge existe, et elle est exacte.** La cellule structurée du dépôt
   possède $2r$ directions de perte exactement plates, par construction. C'est un théorème,
   vérifié numériquement à la précision machine. Son effet sur ce que la descente de gradient
   atteint est, lui, **non mesuré** — et le dépôt a retiré une conclusion antérieure qui allait
   trop vite sur ce point. Démontré au notebook 02
   (`02_les_bras_du_factoriel_et_l_equite.ipynb`).
3. **La taille d'échantillon est l'axe manquant.** À 24 480 transitions pour un LTI à 4 états,
   le problème est sur-déterminé de trois ordres de grandeur : un biais inductif, même
   exactement correct, n'a rien à y acheter. C'est le pire régime possible pour démontrer
   l'utilité d'un prior, et c'est celui dans lequel la mesure a été faite. Développé en §4 de
   ce notebook.

### 0.2 Le plan

| § | Contenu | Nature |
|---|---|---|
| 1 | Pourquoi un predictor générique dérive en rollout libre | démonstration + mesure du dépôt |
| 2 | L'algèbre du bloc rotation-scaling, et son piège numérique | démonstration + vérification |
| 3 | Quand le prior est exactement correct, et quand il ne l'est pas | démonstration + calcul |
| 4 | Ce que la taille d'échantillon change | démonstration + expérience jouet |
| 5 | Récapitulatif des vérifications numériques de ce notebook | inventaire |
| 6 | Où aller ensuite | carte de la série |

Les deux autres faits — l'accessibilité de l'optimum et la dégénérescence de jauge —
ne sont pas redémontrés ici : ils sont cités au fil du texte et prouvés aux notebooks 04 et
02.

### 0.3 Convention de citation, appliquée sans exception

Ce notebook distingue trois statuts, et le statut est écrit à chaque fois :

- **[MESURÉ]** — un chiffre produit par un run du dépôt, avec sa source (`PROTOCOL.md §20.1`,
  `§21.0`, …). Il n'est jamais recalculé ici ; il est cité.
- **[DÉMONTRÉ]** — un énoncé mathématique établi dans le notebook, avec ses hypothèses, et
  vérifié numériquement par la cellule qui suit.
- **[ILLUSTRATIF]** — un chiffre produit par une simulation jouet écrite dans ce notebook. Il
  sert à rendre un mécanisme visible. **Il ne reproduit aucun run** et n'a aucune valeur de
  preuve sur le système réel.

Aucun chiffre de ce notebook n'a d'autre statut. Là où une valeur serait nécessaire au propos
et n'est pas disponible, c'est écrit explicitement.

### 0.4 Dépendances

`numpy`, `matplotlib` et la bibliothèque standard, rien d'autre.

### 0.5 Glossaire, en une passe

Dix termes reviennent dans toute la série ; les définir une fois évite de les redéfinir à
chaque chapitre.

- **JEPA** — déjà introduit en §0 : un encodeur produit un état latent, un predictor l'avance
  sans redécoder. L'objectif combine une perte de prédiction en espace latent et une cible EMA
  (moyenne mobile exponentielle de l'encodeur, gradient arrêté) — construit au notebook 03.
- **Backbone** — l'architecture qui porte la dynamique du predictor : `mlp`, *minimal causal
  transformer*, ou *Mamba-style minimal selective SSM* (`min_ssm`) selon le bras testé.
- **Strate** — le régime de génération des actions dans les campagnes du dépôt : `OU` (bruit
  Ornstein–Uhlenbeck, non conditionné) ou `AC` (actions conditionnées à l'état).
- **Koopman** — un opérateur linéaire **fixe** sur des observables. Un opérateur $\bar A(c_t)$
  qui dépend de l'état n'est jamais appelé Koopman dans ce dépôt : c'est un objet
  **LPV/cocycle** (`REGLES.md §3`).
- **VPT** (*Valid Prediction Time*) — le premier pas de rollout où l'erreur d'état normalisée
  dépasse un seuil déclaré. Défini précisément et utilisé au notebook 03.
- **Pré-enregistrement** — la règle de lecture, les seuils de succès et les budgets de tuning
  sont committés avant que les nombres ne soient vus.
- **`M4`** — la porte de contrôle positif de la campagne pendule (`PROTOCOL.md §21.4` :
  *« M4 is not crossed »*) : sept des huit cellules de la grille `DampedPendulum` (phase D)
  ne passent pas le contrôle positif, cellule par cellule. Le refrain revient verbatim dans
  plusieurs chapitres de la série et se lit facilement comme « le résultat central du dossier
  a échoué ». Ce n'est pas ce qu'il dit : `DampedPendulum` est le système abandonné au
  profit du système conducteur de ce notebook, `CoupledOscillators` (§3), où vivent les
  campagnes comparatives réelles (`PROTOCOL.md §23.4` lève explicitement, pour elles seules,
  l'interdit que pose §21.4). `M4` reste non franchi, publié tel quel ; il mesure simplement
  un système que ce dossier a quitté.
- **Phases (A à E)** — les étapes d'exécution successives du dépôt, nommées par lettre dans
  `PROTOCOL.md` : construction des architectures (phase C), première campagne gated sur
  `DampedPendulum` (phase D, celle que teste `M4`), comparaison complète à budget apparié
  entre backbones (phase E, `results_factorial/`). Seules ces trois-là sont citées dans
  cette série.
- **`OURegime`** — la classe du dépôt (`src/sld/data/regimes.py`) qui implémente la strate
  `OU` définie ci-dessus : bruit Ornstein–Uhlenbeck isotrope, actions en boucle ouverte, non
  conditionnées à l'état.
- **Campagnes A et B** — les deux campagnes comparatives sur `CoupledOscillators` qui portent
  les résultats de cette série (`PROTOCOL.md §23`) : campagne A (racine `results_osc_n/`)
  teste le gain contre la taille d'échantillon à prior exact, campagne B (racine
  `results_osc_delta/`) le teste contre la mauvaise spécification $\delta(c)$ de §3.4. Les
  deux lisent le signe d'une interaction, jamais un seuil de passage (`PROTOCOL.md §23.5`).

### 0.6 Deux utilitaires numériques, réutilisés tout du long

Une exponentielle matricielle par série tronquée, la référence indépendante à laquelle toutes
les formes closes du notebook sont confrontées, et une fonction d'erreur relative, utilisée
dans chaque cellule de vérification qui suit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True, linewidth=110)
plt.rcParams.update({
    "figure.figsize": (10.5, 3.8),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
})

# Exponentielle matricielle par série tronquée : suffisante ici parce que toutes les matrices
# manipulées ont une norme petite (Delta = 0.05, rayons proches de 1). Elle sert de *référence
# indépendante* aux formes closes que nous allons dériver : si l'on validait une forme close
# contre elle-même, on ne validerait rien.
def expm_series(M, n_terms=100):
    """exp(M) par la série de Taylor tronquee. Reference numerique independante."""
    S = np.eye(M.shape[0])
    T = np.eye(M.shape[0])
    for k in range(1, n_terms + 1):
        T = T @ M / k
        S = S + T
    return S


def rel_err(a, b):
    """Erreur relative en norme infinie, avec garde contre le zero."""
    den = max(np.abs(b).max(), 1e-300)
    return float(np.abs(a - b).max() / den)


print("numpy", np.__version__)
print("verification de expm_series contre un cas analytique connu :")
_G = np.array([[0.0, -1.0], [1.0, 0.0]])           # generateur de rotation
_ana = np.array([[np.cos(0.7), -np.sin(0.7)], [np.sin(0.7), np.cos(0.7)]])
print("  |expm_series(0.7*J) - R(0.7)|_inf =", np.abs(expm_series(0.7 * _G) - _ana).max())

## 1. Pourquoi un predictor générique dérive

### 1.1 Deux objectifs qu'on confond, et qui ne sont pas le même

Un predictor latent est une application $f$ qui avance l'état latent d'un pas :
$\hat z_{t+1} = f(\hat z_t)$. On l'entraîne presque toujours sur une **erreur à un pas**

$$\mathcal{L}_1(f) \;=\; \mathbb{E}\big[\;\|f(z_t) - z_{t+1}\|^2\;\big],$$

où $z_t$ est un état **vrai**, fourni par l'encodeur. On s'en sert ensuite pour faire du
**rollout libre** : on part de $\hat z_0 = z_0$, et on itère $f$ sur sa propre sortie, sans
jamais réinjecter d'observation. L'erreur qui compte pour la planification est alors

$$\mathcal{L}_H(f) \;=\; \mathbb{E}\big[\;\|\underbrace{f\circ\cdots\circ f}_{H\ \text{fois}}(z_0) - z_H\|^2\;\big].$$

Ces deux quantités ne mesurent pas la même chose, et le lien entre elles n'est pas une
constante : c'est une **composition**. Établissons-la.

### 1.2 La récurrence de l'erreur

**Hypothèses.** (H1) La dynamique vraie est $z_{t+1} = g(z_t)$. (H2) Le predictor $f$ est
différentiable. (H3) On travaille dans un voisinage où le développement au premier ordre est
valide — c'est-à-dire tant que l'erreur reste petite devant la courbure de $f$. C'est
l'hypothèse qui tombe en premier, et nous y reviendrons.

Posons l'erreur $e_t = \hat z_t - z_t$. Alors

$$e_{t+1} \;=\; f(\hat z_t) - z_{t+1} \;=\; \underbrace{f(\hat z_t) - f(z_t)}_{\text{propagation}} \;+\; \underbrace{f(z_t) - z_{t+1}}_{=:\ \varepsilon_t\ \text{(erreur locale)}}.$$

Le second terme $\varepsilon_t = f(z_t) - g(z_t)$ est exactement ce que $\mathcal{L}_1$
mesure : l'erreur du predictor **partant d'un état vrai**. Le premier terme, sous (H3), vaut
$f(\hat z_t) - f(z_t) = J_t\,e_t + o(\|e_t\|)$ avec $J_t = \partial f/\partial z\,(z_t)$.
D'où la récurrence

$$\boxed{\;e_{t+1} \;=\; J_t\,e_t \;+\; \varepsilon_t\;}$$

C'est une **récurrence linéaire affine**, et elle se déroule exactement. Avec $e_0 = 0$
(on part d'un état vrai) :

$$e_1 = \varepsilon_0,\quad e_2 = J_1\varepsilon_0 + \varepsilon_1,\quad
e_3 = J_2J_1\varepsilon_0 + J_2\varepsilon_1 + \varepsilon_2,\quad\ldots$$

et par récurrence immédiate

$$e_H \;=\; \sum_{k=0}^{H-1}\Big(\prod_{j=k+1}^{H-1} J_j\Big)\,\varepsilon_k .$$

### 1.3 La borne géométrique

Prenons une norme sous-multiplicative $\|\cdot\|$ (norme d'opérateur induite, par exemple),
supposons (H4) $\|J_t\| \le \eta$ pour tout $t$ et (H5) $\|\varepsilon_t\| \le \varepsilon$
pour tout $t$. L'inégalité triangulaire puis la sous-multiplicativité donnent

$$\|e_H\| \;\le\; \sum_{k=0}^{H-1} \Big\|\prod_{j=k+1}^{H-1} J_j\Big\|\,\|\varepsilon_k\|
\;\le\; \sum_{k=0}^{H-1} \eta^{\,H-1-k}\,\varepsilon
\;=\; \varepsilon \sum_{m=0}^{H-1} \eta^{\,m}.$$

La somme géométrique se ferme :

$$\boxed{\;\|e_H\| \;\le\; \varepsilon \cdot \frac{\eta^{H}-1}{\eta-1}\quad (\eta \neq 1),
\qquad \|e_H\| \;\le\; \varepsilon H \quad (\eta = 1).\;}$$

Trois régimes, et ce sont eux qui décident de tout :

| régime | comportement de $\|e_H\|$ | conséquence |
|---|---|---|
| $\eta < 1$ | borné par $\varepsilon/(1-\eta)$ | l'erreur **sature** : le rollout long est utilisable |
| $\eta = 1$ | croît en $\varepsilon H$ | dérive **linéaire** |
| $\eta > 1$ | croît en $\varepsilon\,\eta^{H}/(\eta-1)$ | dérive **géométrique** : rollout inutilisable |

**Ce que cela dit du problème.** Deux predictors avec la même $\mathcal{L}_1$ — la même
$\varepsilon$ — peuvent avoir des $\mathcal{L}_H$ séparées par des ordres de grandeur, si
leurs $\eta$ diffèrent. Minimiser $\mathcal{L}_1$ ne contrôle **pas** $\eta$. C'est le trou
exact dans lequel un prior structurel prétend s'insérer : une paramétrisation où $\eta$ est
lisible, voire bornée par construction.

### 1.4 Quand la borne est serrée, quand elle est lâche

La borne devient une **égalité** quand les deux inégalités utilisées le deviennent :

1. l'inégalité triangulaire est saturée si tous les termes $\big(\prod J\big)\varepsilon_k$
   sont **colinéaires et de même sens** ;
2. $\|J^m v\| = \eta^m\|v\|$ demande que $v$ soit aligné sur la direction de gain maximal de
   $J$, et que cette direction soit **invariante** par $J$.

Les deux tiennent simultanément si $J = \eta\,\mathrm{Id}$ et si les $\varepsilon_k$ sont tous
égaux. C'est le pire cas, et il n'est pas pathologique : une erreur de modèle **systématique**
(un biais, pas un bruit) est exactement un $\varepsilon_k$ constant.

Elle est **très lâche** dans deux cas fréquents :

- $\varepsilon_k$ de moyenne nulle et indépendants : les termes s'additionnent alors en
  variance, pas en norme, et $\mathbb{E}\|e_H\|^2 = \varepsilon^2\sum_m \|J^m\|_F^2$ — la
  racine carrée d'une somme au lieu d'une somme, soit un facteur $\sqrt{H}$ gagné dans le cas
  $\eta = 1$ ;
- $J$ non normale : $\|J\|$ peut valoir beaucoup plus que le rayon spectral $\rho(J)$. La
  borne utilise $\|J\|$, mais le comportement asymptotique est piloté par $\rho(J)$. Entre les
  deux vit la **croissance transitoire** — l'erreur monte puis redescend, et une borne en
  $\|J\|^H$ ne voit jamais la redescente.

Vérifions les trois régimes, et l'écart entre borne au pire cas et réalisation stochastique.

In [ ]:
# --- [ILLUSTRATIF] Simulation de la recurrence e_{t+1} = J e_t + eps_t --------------------
# Deux questions distinctes : (a) la borne de §1.3 est-elle ATTEINTE dans le cas ou §1.4 dit
# qu'elle doit l'etre ? (b) de combien est-elle lache dans les deux cas ou §1.4 dit qu'elle
# doit l'etre ? On ne trace pas "une courbe d'erreur" : on teste deux predictions.

def rot(theta):
    return np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])


def bound(eta, H):
    hs = np.arange(1, H + 1)
    return EPS * (hs if np.isclose(eta, 1.0) else (eta ** hs - 1) / (eta - 1))


def simulate(J, H, mode, n_traj=2000, seed=0):
    """mode='syst' : eps constant et aligne ; mode='alea' : eps de moyenne nulle, meme norme."""
    r = np.random.default_rng(seed)
    E = np.zeros((n_traj, 2))
    v = np.array([1.0, 0.0]) * EPS
    out = []
    for _ in range(H):
        if mode == "syst":
            d = np.broadcast_to(v, E.shape)
        else:
            d = r.standard_normal(E.shape)
            d = EPS * d / np.linalg.norm(d, axis=1, keepdims=True)
        E = E @ J.T + d
        out.append(np.linalg.norm(E, axis=1).mean())
    return np.array(out)


H, EPS = 40, 1e-3
etas = [0.90, 1.00, 1.05, 1.20]
cols = plt.cm.viridis(np.linspace(0.12, 0.82, len(etas)))
hs = np.arange(1, H + 1)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.9))

# (a) le cas ou la borne DOIT etre atteinte : J = eta * Id, erreur locale constante et alignee
print("(a) cas de saturation predit par §1.4 : J = eta*Id, eps constant")
for eta, col in zip(etas, cols):
    sim = simulate(eta * np.eye(2), H, "syst", n_traj=1)
    bd = bound(eta, H)
    axes[0].semilogy(hs, bd, "-", color=col, lw=4.0, alpha=0.30)
    axes[0].semilogy(hs, sim, "-", color=col, lw=1.3, label=rf"$\eta={eta:.2f}$")
    print(f"    eta = {eta:.2f} : ecart relatif max simulation/borne = "
          f"{np.abs(sim - bd).max() / bd.max():.2e}")
axes[0].set(xlabel="horizon $H$", ylabel=r"$\|e_H\|$",
            title="(a) trait epais : borne de §1.3 ; trait fin : simulation")
axes[0].legend(ncol=2)

# (b) les deux cas ou §1.4 predit que la borne est LACHE, a eta fixe
ETA_B = 1.05
bd = bound(ETA_B, H)
variantes = [
    ("borne §1.3", r"borne §1.3", bd, "k", "-", 3.0, 0.35),
    ("J = eta*Id, eps systematique", r"$J=\eta\,$Id, $\varepsilon$ systematique",
     simulate(ETA_B * np.eye(2), H, "syst", 1), "tab:blue", "-", 1.6, 1.0),
    ("J = eta*R(0.35), eps systematique", r"$J=\eta R(0.35)$, $\varepsilon$ systematique",
     simulate(ETA_B * rot(0.35), H, "syst", 1), "tab:orange", "-", 1.6, 1.0),
    ("J = eta*Id, eps de moyenne nulle", r"$J=\eta\,$Id, $\varepsilon$ de moyenne nulle",
     simulate(ETA_B * np.eye(2), H, "alea"), "tab:red", "--", 1.6, 1.0)]
for _plain, lab, y, c, ls, lw, al in variantes:
    axes[1].semilogy(hs, y, ls, color=c, lw=lw, alpha=al, label=lab)
axes[1].set(xlabel="horizon $H$", ylabel=r"$\|e_H\|$",
            title=rf"(b) a $\eta={ETA_B}$ fixe : de combien la borne est-elle lache ?")
axes[1].legend(fontsize=7.5)
plt.tight_layout(); plt.show()

print(f"\n(b) rapport borne / erreur realisee a H = 25, pour eta = {ETA_B} :")
for plain, _lab, y, *_ in variantes[1:]:
    print(f"    {plain:>36} : {bd[24] / y[24]:6.2f} x")

print("\ntaux de composition rho_25 = ||e_25|| / eps (cas de saturation) :")
for eta in etas:
    sim = simulate(eta * np.eye(2), 25, "syst", n_traj=1)
    print(f"    eta = {eta:.2f} : rho_25 = {sim[-1] / EPS:9.2f}   borne = {bound(eta, 25)[-1] / EPS:9.2f}")

**Lecture.** Le panneau (a) teste la prédiction de §1.4 : *la borne est atteinte si
$J = \eta\,\mathrm{Id}$ et si l'erreur locale est un biais constant*. Elle l'est — l'écart
relatif entre simulation et borne est de l'ordre de $10^{-16}$ pour les quatre valeurs de
$\eta$, c'est-à-dire de l'arrondi. La borne de §1.3 n'est donc pas une majoration grossière :
c'est la **solution exacte** de la récurrence dans ce cas. On lit aussi les trois régimes
annoncés : saturation pour $\eta = 0.90$, croissance linéaire pour $\eta = 1.00$, croissance
géométrique au-dessus.

Le panneau (b) teste l'autre moitié de §1.4, à $\eta$ fixé : *la borne est lâche si les
contributions ne s'alignent pas*. Les deux mécanismes prévus sont visibles et **quantifiés**
dans la sortie. Faire tourner $J$ d'un angle de $0.35$ rad par pas suffit à décoller la
courbe de la borne : les termes $J^m\varepsilon$ pointent dans des directions différentes et
s'annulent partiellement. Une erreur locale de moyenne nulle décolle davantage encore :
les contributions s'additionnent alors en variance, pas en norme.

**Ce qu'on ne peut pas conclure de cette figure.** (i) Rien sur un predictor appris : $J$ y
dépend de l'état, et l'hypothèse (H3) — le régime linéaire — tombe dès que l'erreur devient
grande devant la courbure. (ii) Rien sur le classement de deux architectures : $\eta$ et
$\varepsilon$ y varient ensemble, et le produit peut aller dans les deux sens. (iii) Rien sur
le fait qu'un prior structurel *diminue* $\eta$ — c'est une propriété de paramétrisation,
pas une conséquence de cette figure. Ce que la figure établit est plus modeste et plus utile :
**le taux de composition est piloté par un seul scalaire**, et ce scalaire n'apparaît nulle
part dans $\mathcal{L}_1$.

### 1.5 Le fait mesuré : le taux de composition des bras structurés

Le dépôt a couru, en phase D, 40 runs sur le pendule amorti, dans un plan factoriel (toutes
les combinaisons de facteurs) backbone $\times$ structure $\times$ strate de données
(glossaire, §0.5) : ici backbone vaut `mlp` ou `transformer`, et strate vaut OU ou AC.
La grandeur de §1.4 — le taux de composition $\rho = e_{25}/e_1$ — s'y lit
directement, puisque les deux erreurs sont journalisées par run.

**[MESURÉ — recalculé depuis `results_gate_d/positive_control/*/gate.json`
(`measures.e_h`), verdicts `PROTOCOL.md §21.0`, 40 runs, médiane sur 5 seeds]**

| strate | backbone | structure OFF | structure ON | rapport OFF/ON |
|---|---|---|---|---|
| OU | `mlp` / `mlp_k` | 6.24 | 3.03 | (calculé ci-dessous) |
| OU | `transformer` / `transformer_k` | 4.26 | 2.66 | |
| AC | `mlp` / `mlp_k` | 8.58 | 3.92 | |
| AC | `transformer` / `transformer_k` | 3.42 | 3.72 | |

Le suffixe `_k` désigne le bras à structure ON. Les rapports sont recalculés dans la cellule
suivante plutôt que recopiés, pour qu'aucune arithmétique ne soit prise sur parole.

In [ ]:
# --- [MESURE citee, arithmetique derivee ici] ---------------------------------------------
# Source : PROTOCOL.md §21.0 (medianes sur 5 seeds). Les valeurs de rho sont CITEES ;
# seuls les rapports sont calcules.
rho_mesure = {
    ("OU", "mlp"):         (6.24, 3.03),
    ("OU", "transformer"): (4.26, 2.66),
    ("AC", "mlp"):         (8.58, 3.92),
    ("AC", "transformer"): (3.42, 3.72),
}

print(f"{'strate':>7} {'backbone':>12} {'rho OFF':>9} {'rho ON':>8} {'OFF/ON':>8}   verdict")
print("-" * 66)
gains = []
for (strate, bb), (off, on) in rho_mesure.items():
    r = off / on
    gains.append(r)
    verdict = "ON compose plus lentement" if r > 1 else "ON compose PLUS VITE"
    print(f"{strate:>7} {bb:>12} {off:9.2f} {on:8.2f} {r:8.2f}   {verdict}")

favorables = [g for g in gains if g > 1]
print("-" * 66)
print(f"paires ou le bras structure compose plus lentement : {len(favorables)}/4")
print(f"amplitude sur ces paires : {min(favorables):.2f}x a {max(favorables):.2f}x")
print(f"paire defavorable : rapport {min(gains):.2f} (le bras structure compose PLUS VITE)")

**Lecture.** Dans 3 paires sur 4, le bras structuré compose son erreur **1.6× à 2.2× plus
lentement** que son homologue non structuré. C'est le signe attendu si la structure agit sur
$\eta$ — mais le lien n'est pas établi : personne n'a mesuré $\eta$ dans ces runs, seulement
ses conséquences. La quatrième paire (`AC`, `transformer`) va dans l'autre sens, et elle est
publiée telle quelle.

**Une réserve plus basique que les deux confondants ci-dessous.** Ces 8 cellules viennent de
la grille dont le contrôle positif échoue sur 7/8 (`PROTOCOL.md §21.4`, M4 non franchi) — les
4 bras ON affichent 0/5 `learns` ; le taux de composition compare donc surtout des modèles
qui n'ont pas démontrablement appris la dynamique.

**Deux confondants, qui doivent être nommés avant toute lecture causale.**

1. **Asymétrie de taux d'apprentissage** (`PROTOCOL.md §21.2`). Les bras OFF ont reçu un taux
   sélectionné pour eux ; les bras ON ont hérité du taux d'une autre architecture. Un bras
   peut donc apparaître meilleur *ou* pire pour une raison qui n'a rien à voir avec la
   structure. Le dépôt a ouvert un re-run (§21.3) précisément pour réparer ce point ; ce
   notebook ne dispose pas de son verdict.
2. **« Structure ON » est un composite.** Le bras ON ne change pas un seul facteur : il
   remplace la transition **et** ajoute un lift $\psi$ **et** ajoute une projection $\pi$
   ($\psi$ monte $z$ en dimension $2r$ ; $\pi$ redescend). Un effet observé n'est attribuable
   à aucun des trois isolément. C'est exactement le défaut qui
   a invalidé un axe antérieur du dépôt (l'axe $\mu_J$, conservé en matériel historique).

**Statut de l'énoncé, en une phrase.** *Sur ce système, dans ces cellules, avec ces deux
confondants non contrôlés, le signe du taux de composition favorise la structure 3 fois sur
4.* Ce n'est pas « la structure stabilise le rollout ». La différence entre les deux phrases
est tout l'objet de ce notebook.

---

## 2. L'algèbre du bloc rotation-scaling

Le prior utilisé par le dépôt n'est pas une pénalité ajoutée à la perte : c'est une
**paramétrisation**. La transition latente y est bloc-diagonale, chaque bloc étant $2\times 2$
et engendré par

$$G \;=\; \alpha\,I \;+\; \omega\,J,
\qquad I=\begin{pmatrix}1&0\\0&1\end{pmatrix},\quad
J=\begin{pmatrix}0&-1\\1&0\end{pmatrix}.$$

Deux nombres réels par bloc, $(\alpha, \omega)$. Cette section établit tout ce qu'on peut dire
de ce bloc en forme close, parce que c'est ce qui rend le prior **lisible** : après
entraînement, on lit un taux et une pulsation, pas 4 coefficients opaques.

### 2.1 $(I, J)$ est isomorphe à $(1, i)$

**Observation de départ.** $J^2 = \begin{pmatrix}0&-1\\1&0\end{pmatrix}^2
= \begin{pmatrix}-1&0\\0&-1\end{pmatrix} = -I$.

**Proposition.** L'application $\Phi : \alpha + i\omega \mapsto \alpha I + \omega J$ est un
isomorphisme d'algèbres de $\mathbb{C}$ vers la sous-algèbre
$\mathcal{A} = \{\alpha I + \omega J : \alpha,\omega\in\mathbb{R}\} \subset \mathbb{R}^{2\times 2}$.

*Démonstration.* $\Phi$ est clairement $\mathbb{R}$-linéaire et bijective sur son image
($I$ et $J$ sont linéairement indépendantes). Reste la multiplicativité. Pour
$z = \alpha + i\omega$ et $z' = \alpha' + i\omega'$ :

$$\Phi(z)\Phi(z') = (\alpha I + \omega J)(\alpha' I + \omega' J)
= \alpha\alpha' I + (\alpha\omega' + \omega\alpha') J + \omega\omega' J^2 .$$

Avec $J^2 = -I$ :

$$\Phi(z)\Phi(z') = (\alpha\alpha' - \omega\omega')\,I + (\alpha\omega' + \omega\alpha')\,J
= \Phi\big((\alpha\alpha' - \omega\omega') + i(\alpha\omega' + \omega\alpha')\big) = \Phi(zz').\;\square$$

**Conséquence immédiate.** Toute identité analytique valable sur $\mathbb{C}$ et exprimable
par une série entière à coefficients réels se transporte mot pour mot dans $\mathcal{A}$. En
particulier l'exponentielle.

### 2.2 La forme close de $\exp(\Delta G)$

Écrivons la série, et séparons les puissances paires et impaires. Posons $g = \alpha + i\omega$,
de sorte que $G = \Phi(g)$ et $G^n = \Phi(g^n)$.

$$\exp(\Delta G) = \sum_{n\ge 0} \frac{\Delta^n G^n}{n!} = \sum_{n\ge 0}\frac{\Delta^n \Phi(g^n)}{n!}
= \Phi\!\left(\sum_{n\ge 0}\frac{(\Delta g)^n}{n!}\right) = \Phi\big(e^{\Delta g}\big),$$

l'échange somme/$\Phi$ étant licite par continuité et linéarité de $\Phi$ (dimension finie, la
série converge absolument). Or $e^{\Delta g} = e^{\Delta\alpha}e^{i\Delta\omega}
= e^{\Delta\alpha}\big(\cos(\Delta\omega) + i\sin(\Delta\omega)\big)$. Donc

$$\boxed{\;\exp(\Delta G) \;=\; e^{\Delta\alpha}\Big(\cos(\Delta\omega)\,I + \sin(\Delta\omega)\,J\Big)
\;=\; e^{\Delta\alpha}\,R(\Delta\omega),\qquad R(\varphi)=\begin{pmatrix}\cos\varphi & -\sin\varphi\\ \sin\varphi & \cos\varphi\end{pmatrix}.}$$

*(Preuve alternative sans passer par $\mathbb{C}$, pour qui préfère : $G = \alpha I + \omega J$
où $\alpha I$ commute avec tout, donc $\exp(\Delta G) = e^{\Delta\alpha}\exp(\Delta\omega J)$ ;
puis $\exp(\varphi J) = \sum_k \frac{\varphi^{2k}J^{2k}}{(2k)!} + \sum_k\frac{\varphi^{2k+1}J^{2k+1}}{(2k+1)!}
= I\sum_k\frac{(-1)^k\varphi^{2k}}{(2k)!} + J\sum_k\frac{(-1)^k\varphi^{2k+1}}{(2k+1)!}
= \cos\varphi\,I + \sin\varphi\,J$, en utilisant $J^{2k}=(-1)^kI$ et $J^{2k+1}=(-1)^kJ$.)*

### 2.3 Ce que les deux paramètres signifient

La matrice discrète $\bar A = \exp(\Delta G) = e^{\Delta\alpha}R(\Delta\omega)$ est le produit
d'un scalaire positif et d'une rotation. Ses valeurs propres sont
$e^{\Delta\alpha}e^{\pm i\Delta\omega}$, donc :

$$\boxed{\;r \;=\; \rho(\bar A) \;=\; \|\bar A\|_2 \;=\; e^{\Delta\alpha},
\qquad \varphi \;=\; \Delta\omega. \;}$$

Trois remarques qui font tout l'intérêt de la paramétrisation :

1. **Le rayon spectral est en forme close et exact.** Il n'est ni estimé ni borné : il est
   *lu*. Cela relie directement §1 : le $\eta$ dont dépendait la composition de l'erreur est
   ici $e^{\Delta\alpha}$, un scalaire par bloc, disponible à chaque pas d'entraînement.
2. **$\bar A$ est normale** ($\bar A^\top \bar A = e^{2\Delta\alpha}I$), donc
   $\|\bar A\|_2 = \rho(\bar A)$ : il n'y a **pas** de croissance transitoire non normale à
   l'intérieur d'un bloc. La borne de §1.3 y est donc exacte, pas conservatrice. C'est un
   argument en faveur de la structure qui est souvent tu, et qui est propre.
3. **$\alpha$ et $\omega$ sont découplés** : $\alpha$ est un taux d'amortissement (dimension
   $\mathrm{temps}^{-1}$), $\omega$ une pulsation (dimension $\mathrm{temps}^{-1}$ aussi, mais
   angulaire). Le modèle continu sous-jacent $\dot h = Gh$ a pour solutions
   $e^{\alpha t}(\cos\omega t, \sin\omega t)$ : une spirale.

**Choix de design, et l'alternative écartée.** On aurait pu paramétrer directement les 4
coefficients de $\bar A$ et *pénaliser* $\rho(\bar A)$. Cette alternative est écartée pour
trois raisons : (i) $\rho$ n'est pas une fonction lisse des coefficients aux valeurs propres
multiples, donc son gradient est mal défini là où il compte ; (ii) une pénalité est un
compromis réglable, donc un hyperparamètre de plus dans un plan factoriel qui en a déjà
beaucoup ; (iii) surtout, une pénalité ne garantit rien — la contrainte est **architecturale**
ici, ce qui rend l'énoncé « la classe contient exactement ces dynamiques » vérifiable au lieu
d'espéré. C'est la ligne du dépôt : `structure` est un facteur d'architecture, jamais un terme
numérique dans la perte.

Vérifions la forme close contre l'exponentielle par série, qui n'en sait rien.

In [ ]:
# --- [DEMONTRE -> verifie] Forme close de exp(Delta G) contre la serie ---------------------
I2 = np.eye(2)
J2 = np.array([[0.0, -1.0], [1.0, 0.0]])
DELTA = 0.05                       # pas d'integration, celui du depot pour les oscillateurs

print("J @ J =\n", J2 @ J2, "\n-> J^2 = -I :", np.allclose(J2 @ J2, -I2), "\n")


def A_bar_closed(alpha, omega, dt=DELTA):
    """exp(dt * (alpha I + omega J)) en forme close."""
    return np.exp(dt * alpha) * (np.cos(dt * omega) * I2 + np.sin(dt * omega) * J2)


cases = [(-0.30, 2.10), (0.70, -1.30), (0.0, 3.0), (-5.0, 0.0), (0.0, 0.0), (2.0, 40.0)]
print(f"{'alpha':>8} {'omega':>8} | {'|close - serie|_inf':>21} | {'r = e^(D a)':>12} {'rho(Abar)':>11} {'|Abar|_2':>10}")
print("-" * 92)
for a, w in cases:
    G = a * I2 + w * J2
    Ac = A_bar_closed(a, w)
    As = expm_series(DELTA * G)
    rad = max(abs(np.linalg.eigvals(Ac)))
    print(f"{a:8.2f} {w:8.2f} | {np.abs(Ac - As).max():21.3e} | "
          f"{np.exp(DELTA*a):12.9f} {rad:11.9f} {np.linalg.svd(Ac, compute_uv=False)[0]:10.7f}")

# normalite : Abar^T Abar doit valoir e^{2 D alpha} I
a, w = -0.30, 2.10
Ac = A_bar_closed(a, w)
print("\nnormalite du bloc : |Abar^T Abar - e^(2 D alpha) I|_inf =",
      np.abs(Ac.T @ Ac - np.exp(2 * DELTA * a) * I2).max())

**Lecture.** La forme close et la série tronquée coïncident à $10^{-16}$ près, y compris au
cas extrême $\omega = 40$ (soit $\Delta\omega = 2$ rad par pas, presque un tiers de tour) et
au cas dégénéré $(\alpha,\omega)=(0,0)$ où $\bar A = I$. Les trois colonnes de droite
confirment l'identité annoncée : rayon spectral, norme spectrale et $e^{\Delta\alpha}$ sont le
**même nombre** à la précision machine — c'est la normalité du bloc, vérifiée en dernière
ligne.

**Ce qu'on ne peut pas conclure.** Rien sur la stabilité de l'entraînement : $r = e^{\Delta\alpha}$
est exact mais **non borné** si $\alpha$ est laissé libre, ce qui est le choix du dépôt
(`PROTOCOL.md §13.2` : *« no `-exp(nu)`, no `-softplus` »*). Un $r > 1$ sous un rollout fermé
de $H$ pas amplifie le gradient par $r^H$ — à $r = 1.20$ et $H = 25$, un facteur $95$. La
lisibilité du rayon et sa **maîtrise** sont deux choses différentes.

### 2.4 L'intégrale ZOH, en forme close

Un predictor latent doit absorber une action. Le modèle continu est

$$\dot h(t) \;=\; G\,h(t) \;+\; B\,u(t).$$

**Hypothèse de discrétisation (ZOH, *zero-order hold*).** L'action est constante sur
l'intervalle : $u(t) = u_k$ pour $t \in [t_k, t_k+\Delta)$. C'est l'hypothèse standard en
contrôle échantillonné, et elle est *exacte* — pas approchée — si le contrôleur tient
effectivement sa commande entre deux pas, ce qui est le cas dans un environnement à pas de
temps discret.

La formule de variation de la constante donne alors, exactement :

$$h(t_k+\Delta) \;=\; e^{\Delta G} h(t_k) \;+\; \left(\int_0^{\Delta} e^{sG}\,ds\right) B\,u_k
\;=:\; \bar A\, h_k + \bar B\, u_k ,
\qquad \bar A = e^{\Delta G},\quad \bar B = \Phi B,\quad \Phi := \int_0^{\Delta} e^{sG}\,ds .$$

*(Dérivation : posons $y(s) = e^{-sG}h(t_k+s)$. Alors
$y'(s) = -Ge^{-sG}h + e^{-sG}(Gh + Bu_k) = e^{-sG}Bu_k$, en utilisant que $G$ et $e^{-sG}$
commutent. On intègre de $0$ à $\Delta$ : $y(\Delta) - y(0) = \big(\int_0^\Delta e^{-sG}ds\big)Bu_k$,
puis on multiplie à gauche par $e^{\Delta G}$ et on change de variable $s \to \Delta - s$.)*

**Calcul de $\Phi$ dans la base $(I,J)$.** Par l'isomorphisme de §2.1, $\Phi = \Phi_{\mathcal{A}}$
où l'on calcule dans $\mathbb{C}$ avec $g = \alpha + i\omega$ :

$$\int_0^{\Delta} e^{sg}\,ds \;=\; \frac{e^{\Delta g}-1}{g}\qquad (g \neq 0).$$

On rationalise en multipliant par $\bar g = \alpha - i\omega$ :

$$\frac{e^{\Delta g}-1}{g} \;=\; \frac{(e^{\Delta g}-1)(\alpha - i\omega)}{\alpha^2+\omega^2}.$$

Avec $e^{\Delta g} - 1 = \big(e^{\Delta\alpha}\cos\Delta\omega - 1\big) + i\,e^{\Delta\alpha}\sin\Delta\omega$,
le produit se développe et l'on identifie partie réelle (coefficient de $I$) et partie
imaginaire (coefficient de $J$) :

$$\boxed{\;\Phi \;=\; \frac{1}{\alpha^2+\omega^2}\Big[
\underbrace{\big(\alpha(e^{\Delta\alpha}\cos\Delta\omega - 1) + \omega\,e^{\Delta\alpha}\sin\Delta\omega\big)}_{\text{coefficient de }I}\;I
\;+\;
\underbrace{\big(\alpha\,e^{\Delta\alpha}\sin\Delta\omega - \omega(e^{\Delta\alpha}\cos\Delta\omega - 1)\big)}_{\text{coefficient de }J}\;J\Big].}$$

### 2.5 La singularité en $(0,0)$ est *removable* — et c'est un vrai piège numérique

Le dénominateur $\alpha^2+\omega^2 = |g|^2$ s'annule en $(\alpha,\omega)=(0,0)$. **Ce n'est pas
une singularité de $\Phi$**, seulement de son écriture. En effet, la série entière

$$\frac{e^{\Delta g}-1}{g} \;=\; \frac{1}{g}\sum_{n\ge 1}\frac{(\Delta g)^n}{n!}
\;=\; \sum_{n\ge 1}\frac{\Delta^n g^{\,n-1}}{n!}
\;=\; \Delta + \frac{\Delta^2 g}{2} + \frac{\Delta^3 g^2}{6} + \cdots$$

est **entière** en $g$ : elle converge partout, et vaut $\Delta$ en $g=0$ — ce qui est le
résultat attendu, puisque $\int_0^\Delta e^{s\cdot 0}ds = \Delta$. La singularité est donc
**effaçable** au sens de l'analyse complexe : la fonction se prolonge holomorphiquement.

**Mais l'annulation numérique, elle, ne s'efface pas.** Quand $|g| \to 0$, le numérateur
$e^{\Delta g}-1$ tend vers $0$ **et** le dénominateur $|g|^2$ aussi. On divise donc deux
quantités qui ont perdu leurs chiffres significatifs par cancellation. Le mécanisme précis :
$e^{\Delta\alpha}\cos(\Delta\omega) - 1$ est calculé comme une différence de deux nombres
proches de $1$, ce qui détruit $\log_{10}(1/|\Delta g|)$ décimales avant même la division.

**Correctif, et c'est celui à implémenter :** basculer sur la série tronquée quand
$|g| = \sqrt{\alpha^2+\omega^2}$ passe sous un seuil. Le seuil se choisit en égalisant les deux
sources d'erreur : la troncature à l'ordre $n$ laisse $O(\Delta^{n+1}|g|^{n}/(n+1)!)$, la
cancellation coûte $O(\epsilon_{\text{mach}}/(\Delta|g|))$ en relatif. Avec 8 termes et
$\Delta \sim 10^{-2}$, un seuil $|g| \sim 10^{-8}$ place les deux sous $10^{-15}$. Mesurons-le
plutôt que de l'affirmer.

In [ ]:
# --- [DEMONTRE -> verifie] Forme close de Phi, et le piege de la cancellation --------------
from math import factorial


def Phi_series(alpha, omega, dt=DELTA, n_terms=10):
    """Phi par serie entiere : sum_{n>=1} dt^n G^{n-1} / n!  -- valable partout, exacte en 0."""
    G = alpha * I2 + omega * J2
    S = dt * I2
    T = np.eye(2)
    for n in range(1, n_terms):
        T = T @ G
        S = S + dt ** (n + 1) / factorial(n + 1) * T
    return S


def Phi_naive(alpha, omega, dt=DELTA):
    """Forme close BRUTE : correcte en arithmetique exacte, instable pres de (0,0)."""
    n2 = alpha ** 2 + omega ** 2
    E, co, si = np.exp(dt * alpha), np.cos(dt * omega), np.sin(dt * omega)
    coef_I = alpha * (E * co - 1.0) + omega * E * si
    coef_J = alpha * E * si - omega * (E * co - 1.0)
    return (coef_I * I2 + coef_J * J2) / n2


def Phi_closed(alpha, omega, dt=DELTA, tol=1e-8):
    """Forme close AVEC bascule sur la serie : c'est la version a implementer."""
    if alpha ** 2 + omega ** 2 < tol ** 2:
        return Phi_series(alpha, omega, dt)
    return Phi_naive(alpha, omega, dt)


def Phi_quad(alpha, omega, dt=DELTA, n=20001, n_terms=40):
    """Reference independante : quadrature trapezoidale fine de int_0^dt exp(sG) ds.

    L'exponentielle est evaluee par serie tronquee, vectorisee sur les n points de quadrature.
    40 termes suffisent tres largement : ||s G|| <= dt * |g| reste petit devant 1.
    """
    s = np.linspace(0.0, dt, n)
    G = alpha * I2 + omega * J2
    Gs = s[:, None, None] * G                       # (n, 2, 2)
    S = np.broadcast_to(np.eye(2), (n, 2, 2)).copy()
    T = S.copy()
    for k in range(1, n_terms + 1):
        T = np.einsum("nij,njk->nik", T, Gs) / k
        S = S + T
    return np.trapezoid(S, s, axis=0)


print("Verification de la forme close contre une quadrature fine (20001 points) :")
print(f"{'alpha':>10} {'omega':>10} | {'|close - quad|_inf':>19} | {'|naive - quad|_inf':>19}")
print("-" * 68)
for a, w in [(-0.30, 2.10), (0.70, -1.30), (0.0, 3.0), (-5.0, 0.0)]:
    q = Phi_quad(a, w)
    print(f"{a:10.2f} {w:10.2f} | {np.abs(Phi_closed(a, w) - q).max():19.3e} |"
          f" {np.abs(Phi_naive(a, w) - q).max():19.3e}")

# Validation INDEPENDANTE du bras "serie", a |g| ~ 1 ou la quadrature est fiable : sans cela,
# comparer la bascule a la serie reviendrait a comparer la serie a elle-meme.
for a, w in [(0.6, 0.8), (-0.7, 0.7)]:
    print(f"  |Phi_series - Phi_quad|_inf en (alpha,omega)=({a},{w}) :"
          f" {np.abs(Phi_series(a, w) - Phi_quad(a, w)).max():.3e}")

print("\nApproche de la singularite (alpha = omega = t), erreur RELATIVE :")
print(f"{'|g|':>12} | {'naive':>12} | {'close (bascule)':>16} | {'reference':>13}")
print("-" * 62)
mags, err_naive, err_close = [], [], []
for t in np.logspace(-1, -12, 12):
    a = w = t / np.sqrt(2.0)
    ref = Phi_series(a, w, n_terms=16)          # exacte a cette echelle : |g| petit
    en = rel_err(Phi_naive(a, w), ref)
    ec = rel_err(Phi_closed(a, w), ref)
    # plancher d'affichage a eps_mach : sous le seuil de bascule la version corrigee EST
    # la serie de reference, donc son "erreur" mesuree est nulle par construction.
    mags.append(t); err_naive.append(max(en, 2.2e-16)); err_close.append(max(ec, 2.2e-16))
    print(f"{t:12.1e} | {en:12.2e} | {ec:16.2e} | {'(reference)':>13}")

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.loglog(mags, err_naive, "o-", label=r"forme close brute $\;(e^{\Delta g}-1)\bar g/|g|^2$")
ax.loglog(mags, err_close, "s-", label=r"avec bascule serie pour $|g|<10^{-8}$ (plancher affiche : $\epsilon_{\rm mach}$)")
ax.axvline(1e-8, color="k", ls=":", lw=1, label=r"seuil de bascule")
ax.axhline(2.2e-16, color="gray", ls="--", lw=1, label=r"$\epsilon_{\rm mach}$")
ax.set(xlabel=r"$|g| = \sqrt{\alpha^2+\omega^2}$", ylabel=r"erreur relative sur $\Phi$",
       title=r"Singularite effacable en $(0,0)$ : analytiquement inoffensive, numeriquement non")
ax.invert_xaxis(); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

**Lecture.** La forme close brute et la quadrature coïncident à $4.7\times10^{-14}$ –
$5.8\times10^{-13}$ près tant que $|g|$ est d'ordre 1 (le pire des quatre cas testés) — et cet
écart résiduel est celui de la **quadrature**, pas de la formule : le trapèze sur 20 001 points
laisse un $O(h^2)$ de cet ordre. La courbe log-log montre ensuite
ce que la formule brute fait quand on s'approche de $(0,0)$ : l'erreur relative croît à peu
près comme $1/|g|$ et atteint $7\cdot 10^{-4}$ à $|g| = 10^{-12}$ — il ne reste alors que trois
chiffres significatifs — puis la formule rend `nan` en $|g| = 0$ exactement (division $0/0$).

**Une précaution de lecture sur la colonne de droite.** Sous le seuil de bascule, la version
corrigée *est* la série, et la référence aussi : le zéro affiché est donc une tautologie, pas
une mesure. C'est pourquoi le bras « série » est validé séparément, contre la quadrature, à
$|g| \sim 1$ où celle-ci est fiable (lignes imprimées au-dessus du tableau, $\sim 2.5\cdot 10^{-14}$).
Sans cette précaution, on aurait « vérifié » la série contre elle-même.

La bascule sur la série tronquée ramène l'erreur au niveau de $\epsilon_{\text{mach}}$ sur
toute la plage. C'est un correctif de trois lignes, et son absence est le genre de défaut qui
ne se manifeste qu'après plusieurs milliers d'updates — au moment où un bloc, en s'amortissant,
voit son $(\alpha,\omega)$ traverser le voisinage de l'origine.

**Confrontation au code, sans l'exécuter ici.** `src/sld/predictors/blocks2x2.py::rotation_scaling_zoh` implémente ce même correctif (vérifié sur le fichier, pas exécuté dans ce notebook numpy-pur) : bascule entre forme close et série tronquée selon un seuil sur $\Delta^2(\alpha^2+\omega^2)$, avec un commentaire de code qui nomme explicitement la même singularité *removable*. La parité de niveau 3 contre ce module (la cellule qui importerait `sld` et confronterait les deux versions par `np.allclose`) vit aux notebooks 02, 03 et 05 ; ce chapitre reste numpy-pur par choix (§6.3) et se limite donc à citer la ligne plutôt qu'à l'exécuter.

**Ce qu'on ne peut pas conclure.** Que ce chemin ait été traversé dans les runs du dépôt :
la trajectoire de $(\alpha, \omega)$ par bloc n'est pas journalisée, donc l'événement n'est
**ni observé ni exclu**. C'est un risque identifié, pas un bug mesuré.

## 3. Quand le prior est exactement correct, et quand il ne l'est pas

Un prior structurel n'est ni vrai ni faux dans l'absolu : il est **bien ou mal spécifié pour
un système donné**. Cette section construit un système où l'on peut faire varier
continûment le degré de mauvaise spécification, en tournant un seul bouton physique. C'est
l'outil qui manque à la plupart des discussions sur les biais inductifs : sans axe de
mauvaise spécification, on ne compare que des anecdotes.

### 3.1 Le système : deux masses couplées

Deux masses $m_1 = m_2 = 1$, reliées au bâti par des ressorts $k_1 = 1$ et $k_2 = 1.5$, et
l'une à l'autre par un ressort de couplage $k_c = 0.5$. Amortissement visqueux de coefficient
$c$ sur chaque masse. Les équations de Newton donnent

$$\ddot q_1 = -(k_1+k_c)q_1 + k_c q_2 - c\,\dot q_1, \qquad
\ddot q_2 = k_c q_1 - (k_2+k_c)q_2 - c\,\dot q_2 .$$

En posant $q = (q_1,q_2)^\top$ et $K = \begin{pmatrix}k_1+k_c & -k_c\\ -k_c & k_2+k_c\end{pmatrix}$,
cela s'écrit $\ddot q = -Kq - c\dot q$, soit en état $x = (q, \dot q)^\top \in \mathbb{R}^4$ :

$$\dot x = A x, \qquad A = \begin{pmatrix} 0 & I \\ -K & -cI\end{pmatrix}.$$

In [ ]:
# --- [DEMONTRE -> trace] Le systeme conducteur : deux masses couplees, deux regimes de c ----
# K et A_state(c) sont definis ici et reutilises tels quels par le reste du chapitre (§3.2 et
# suivants) : c'est le premier trace de trajectoire de toute la serie sur ce systeme -- REGLES.md
# §8 l'exige avant tout diagnostic. On voit ce que font les deux masses avant de raisonner sur
# leur spectre.
k1, kc, k2 = 1.0, 0.5, 1.5
K = np.array([[k1 + kc, -kc], [-kc, k2 + kc]])


def A_state(c):
    """Matrice d'etat A(c) du systeme conducteur (§3.1) : dx/dt = A(c) x, x = (q1, q2, q1., q2.)."""
    Z, Id = np.zeros((2, 2)), np.eye(2)
    return np.block([[Z, Id], [-K, -c * Id]])


DT_TRAJ, N_STEPS = 0.05, 300
X0 = np.array([1.0, -0.6, 0.0, 0.0])   # deplacement initial, vitesses nulles


def trajectory(c, x0=X0, dt=DT_TRAJ, n_steps=N_STEPS):
    """Propage x0 par pas exacts exp(dt*A(c)), calcules par la reference independante de §0.6."""
    A_d = expm_series(dt * A_state(c))
    xs = [x0]
    for _ in range(n_steps):
        xs.append(A_d @ xs[-1])
    return np.array(xs)


C_LIGHT, C_DAMPED = 0.10, 3.00       # c = 3.00 : point le plus mal specifie de la zone monotone
                                       # de §3.4 (delta = 1.029, table de la cellule suivant §3.4)
traj_light, traj_damped = trajectory(C_LIGHT), trajectory(C_DAMPED)
ts = DT_TRAJ * np.arange(N_STEPS + 1)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=True)
for ax, traj, c_val, titre in [
    (axes[0], traj_light, C_LIGHT, f"(a) $c={C_LIGHT}$ -- sous-amorti, prior EXACT ($\\delta=0$)"),
    (axes[1], traj_damped, C_DAMPED, f"(b) $c={C_DAMPED}$ -- suramorti ($\\delta=1.029$, §3.4)"),
]:
    ax.plot(ts, traj[:, 0], label="$q_1(t)$")
    ax.plot(ts, traj[:, 1], label="$q_2(t)$")
    ax.axhline(0, color="k", lw=0.6)
    ax.set(xlabel="temps (s)", ylabel="position" if ax is axes[0] else "", title=titre)
    ax.legend()
plt.tight_layout(); plt.show()

**Lecture.** Panneau (a), à $c=0.10$ (le régime que la campagne D3 mesure, contrôle D3
§3.5) : les deux masses oscillent en s'amortissant lentement, le régime **sous-amorti**, où le
spectre de $A(c)$ est deux paires conjuguées et où le prior rotation-scaling est exactement
correct ($\delta=0$, démontré en §3.4). Panneau (b), même système, avec $c=3.00$ (le point le
plus mal spécifié de la zone monotone de §3.4, $\delta=1.029$) : les deux masses reviennent à
l'équilibre sans osciller, le régime **suramorti**. Le même scalaire physique $c$ qui règle
l'axe $\delta(c)$ de §3.4 pilote donc directement ce qu'on voit ici : passer d'un régime à
l'autre change qualitativement la trajectoire, pas seulement un chiffre abstrait dans un
tableau.

**Ce qu'on ne peut pas conclure de cette figure.** (i) Rien sur ce qu'un predictor appris ferait
de ce système : ce sont les positions vraies $q_1(t), q_2(t)$, propagées par $\exp(\Delta A(c))$
exact, sans aucun rollout ni erreur de modèle. (ii) Rien sur *pourquoi* $c=3.00$ est mal
spécifié pour la classe rotation-scaling : c'est la question du théorème de représentabilité,
§3.3, qui vient ensuite. (iii) Que les deux masses semblent visuellement « similaires » en
régime suramorti ne dit rien sur la distance spectrale $\delta$ : deux systèmes suramortis
différents peuvent avoir des $\delta$ très différents, comme le montre la courbe de §3.4.

### 3.2 Découplage modal : le spectre de $A$ à partir de celui de $K$

**Proposition.** Soit $\kappa$ une valeur propre de $K$. Alors les valeurs propres associées
de $A$ sont les racines de

$$\boxed{\;\lambda^2 + c\,\lambda + \kappa = 0.\;}$$

*Démonstration.* $K$ est symétrique réelle, donc orthogonalement diagonalisable :
$K = V\Lambda V^\top$ avec $V^\top V = I$ et $\Lambda = \mathrm{diag}(\kappa_1,\kappa_2)$.
Posons le changement de variables en **coordonnées modales**
$\tilde q = V^\top q$, $\tilde v = V^\top \dot q$, c'est-à-dire
$\tilde x = T x$ avec $T = \mathrm{diag}(V^\top, V^\top)$, matrice orthogonale de $\mathbb{R}^4$.
Alors

$$T A T^\top = \begin{pmatrix} V^\top & 0\\ 0 & V^\top\end{pmatrix}
\begin{pmatrix} 0 & I\\ -K & -cI\end{pmatrix}
\begin{pmatrix} V & 0\\ 0 & V\end{pmatrix}
= \begin{pmatrix} 0 & V^\top V\\ -V^\top K V & -c\,V^\top V\end{pmatrix}
= \begin{pmatrix} 0 & I\\ -\Lambda & -cI\end{pmatrix}.$$

Comme $\Lambda$ est diagonale, ce système est **découplé** : chaque paire
$(\tilde q_i, \tilde v_i)$ obéit à

$$\begin{pmatrix}\dot{\tilde q}_i\\ \dot{\tilde v}_i\end{pmatrix}
= \begin{pmatrix}0 & 1\\ -\kappa_i & -c\end{pmatrix}
\begin{pmatrix}\tilde q_i\\ \tilde v_i\end{pmatrix},$$

soit l'oscillateur scalaire amorti $\ddot{\tilde q}_i + c\,\dot{\tilde q}_i + \kappa_i \tilde q_i = 0$.
Son polynôme caractéristique est $\lambda^2 + c\lambda + \kappa_i$. Le spectre de $A$ est la
réunion des spectres des deux blocs $2\times 2$, car $A$ est semblable (par une similitude
**orthogonale**, donc parfaitement conditionnée) à une matrice bloc-diagonale. $\square$

**Corollaire — le seuil de suramortissement.** Les racines sont
$\lambda_{\pm} = \frac{-c \pm \sqrt{c^2-4\kappa}}{2}$. Elles forment une paire complexe
conjuguée si $c^2 < 4\kappa$ (mode **sous-amorti**), une racine réelle double si $c^2 = 4\kappa$
(**critique**), deux réelles distinctes si $c^2 > 4\kappa$ (**suramorti**). Le seuil est donc

$$c^*(\kappa) = 2\sqrt{\kappa}.$$

**Pourquoi ce système et pas un autre.** Trois raisons, et une alternative écartée.
(i) Il est **linéaire**, donc la classe rotation-scaling peut le contenir *exactement* — c'est
la condition sine qua non pour distinguer « le prior est faux » de « l'optimisation échoue »,
la question que mesure le notebook 04 sur ce même système (contrôle D3, présenté en §3.5). (ii) La similitude modale est **orthogonale**, donc $\mathrm{cond}(V) = 1$ : aucun effet
de mauvais conditionnement ne vient contaminer la mesure. (iii) Un seul scalaire $c$ fait
passer continûment de « prior exact » à « prior faux ». *L'alternative écartée* : le pendule
amorti, non linéaire, sur lequel l'axe ON/OFF du dépôt a effectivement été mesuré. Un prior
LTI y est faux **par construction**, donc un résultat négatif n'y départage rien — et c'est
précisément la critique que le dépôt s'adresse à lui-même.

In [ ]:
# --- [DEMONTRE -> verifie] Decouplage modal et seuils de suramortissement ------------------
# K et A_state(c) sont deja definis en §3.1 : reutilises tels quels.
kappas = np.sort(np.linalg.eigvalsh(K))
c_star = 2.0 * np.sqrt(kappas)

print("K =\n", K)
print("\nvaleurs propres de K   : kappa =", np.round(kappas, 6))
print("seuils c* = 2 sqrt(kappa) :", np.round(c_star, 6))


def canon_sort(ev, nd=10):
    """Tri canonique de valeurs propres complexes : partie reelle ARRONDIE, puis imaginaire.
    L'arrondi est indispensable : sans lui, deux valeurs propres de meme partie reelle
    mathematiquement sont departagees par un ecart de 1e-17 et l'ordre devient arbitraire."""
    ev = np.asarray(ev)
    return ev[np.lexsort((ev.imag, np.round(ev.real, nd)))]


print("\nVerification : le spectre de A coincide avec les racines de lambda^2 + c lambda + kappa")
print(f"{'c':>6} | {'spectre de A (numerique)':>44} | {'ecart aux racines modales':>25}")
print("-" * 84)
for c in [0.10, 1.00, 2.30, 3.00, 4.00]:
    ev = canon_sort(np.linalg.eigvals(A_state(c)))
    modal = []
    for kap in kappas:
        disc = complex(c * c - 4 * kap) ** 0.5
        modal += [(-c + disc) / 2, (-c - disc) / 2]
    modal = canon_sort(np.array(modal))
    print(f"{c:6.2f} | {np.array2string(ev, precision=4, suppress_small=True):>44} |"
          f" {np.abs(ev - modal).max():25.3e}")

**Lecture.** Le spectre calculé numériquement sur la matrice $4\times4$ et les racines des
deux polynômes modaux coïncident à $10^{-15}$ près pour toutes les valeurs de $c$ testées : le
découplage démontré est exact, et l'on peut désormais raisonner sur deux oscillateurs
scalaires indépendants au lieu d'une matrice $4\times4$. Les seuils valent
$c^*_1 = 2\sqrt{1.191} \approx 2.183$ et $c^*_2 = 2\sqrt{2.309} \approx 3.039$.

**Ce qu'on ne peut pas conclure.** Que le prior soit approprié : le découplage dit seulement
que la dynamique se factorise en deux sous-systèmes $2\times2$. Il ne dit rien sur la forme de
ces blocs, qui est exactement la question suivante.

### 3.3 Théorème de représentabilité

**Énoncé.** Un bloc rotation-scaling de générateur $G = \alpha I + \omega J$ a pour spectre
$\{\alpha + i\omega,\ \alpha - i\omega\}$ en temps continu (et
$\{e^{\Delta\alpha}e^{\pm i\Delta\omega}\}$ en temps discret). Un tel bloc peut donc
représenter :

- une paire complexe conjuguée quelconque ($\omega \neq 0$) ;
- une valeur propre **réelle double** ($\omega = 0$, spectre $\{\alpha,\alpha\}$).

Il ne peut **pas** représenter deux valeurs propres réelles distinctes. De plus, si
$\lambda_a \neq \lambda_b$ sont réelles, la distance spectrale minimale d'un bloc à la paire
$\{\lambda_a,\lambda_b\}$, au sens du pire appariement (*bottleneck*), vaut **exactement**

$$d(\lambda_a,\lambda_b) \;=\; \min_{\alpha,\omega\in\mathbb{R}}\;
\max\big(|\lambda_a - (\alpha+i\omega)|,\ |\lambda_b - (\alpha-i\omega)|\big)
\;=\; \frac{|\lambda_a - \lambda_b|}{2},$$

atteinte en $\alpha = \frac{\lambda_a+\lambda_b}{2}$, $\omega = 0$.

*Démonstration.* Posons $z = \alpha + i\omega \in \mathbb{C}$ ; quand $(\alpha,\omega)$ parcourt
$\mathbb{R}^2$, $z$ parcourt $\mathbb{C}$ tout entier. Le second terme se réécrit

$$|\lambda_b - (\alpha - i\omega)| \;=\; |\lambda_b - \bar z| \;=\; \big|\overline{\lambda_b - \bar z}\big|
\;=\; |\bar\lambda_b - z|.$$

Le problème devient donc : minimiser sur $z \in \mathbb{C}$ le maximum des distances de $z$ à
**deux points fixes du plan**, $\lambda_a$ et $\mu := \bar\lambda_b$. Or pour tout $z$,

$$\max\big(|\lambda_a - z|,\ |\mu - z|\big)
\;\ge\; \tfrac{1}{2}\big(|\lambda_a - z| + |z - \mu|\big)
\;\ge\; \tfrac{1}{2}|\lambda_a - \mu|,$$

la première inégalité parce qu'un maximum majore la moyenne, la seconde par inégalité
triangulaire. La borne $\frac{1}{2}|\lambda_a-\mu|$ est atteinte en $z^\star = \frac{\lambda_a+\mu}{2}$,
où les deux distances valent exactement $\frac{1}{2}|\lambda_a-\mu|$. Donc

$$d \;=\; \tfrac{1}{2}\,\big|\lambda_a - \bar\lambda_b\big|.$$

Deux cas :
- **$\lambda_b = \bar\lambda_a$** (paire conjuguée) : $\bar\lambda_b = \lambda_a$, donc $d = 0$
  — le bloc représente la paire **exactement**, en prenant $\alpha = \Re\lambda_a$,
  $\omega = \Im\lambda_a$ ;
- **$\lambda_a,\lambda_b$ réelles** : $\bar\lambda_b = \lambda_b$, donc
  $d = \frac{1}{2}|\lambda_a - \lambda_b|$, atteinte en $z^\star = \frac{\lambda_a+\lambda_b}{2}$
  réel, c'est-à-dire $\omega = 0$. $\square$

**Où l'énoncé tombe.** (i) Il mesure une distance **spectrale**, pas une distance entre
opérateurs : deux matrices peuvent avoir le même spectre et des comportements transitoires
très différents (non-normalité). (ii) Il suppose l'appariement pire cas ; avec une somme au
lieu d'un max, l'optimum est le même ici, mais ce n'est pas automatique en dimension
supérieure. (iii) Il ne dit rien sur ce qu'un **entraînement** trouve : c'est une borne
d'approximation, pas d'optimisation. Cet écart entre approximation et optimisation est
entièrement le sujet du notebook 04.

In [ ]:
# --- [DEMONTRE -> verifie] La distance minimale a un bloc vaut |la - conj(lb)| / 2 ---------
# On ne fait pas confiance a la formule : on minimise numeriquement sur une grille fine + un
# raffinement local, et on compare a la forme close.

def bottleneck_obj(z, la, lb):
    return max(abs(la - z), abs(lb - np.conj(z)))


def brute_min(la, lb, n=241, rounds=8):
    """Minimisation directe de max(|la - z|, |lb - conj(z)|) par raffinement de grille.

    Le point de depart est (0, 0) et NON le minimiseur analytique : sinon la "verification"
    partirait de la reponse et ne verifierait rien.
    """
    ac, wc = 0.0, 0.0
    half = 4.0 + abs(la) + abs(lb)
    best = (np.inf, ac, wc)
    for _ in range(rounds):
        As = np.linspace(ac - half, ac + half, n)
        Ws = np.linspace(wc - half, wc + half, n)
        for a in As:
            zs = a + 1j * Ws
            vals = np.maximum(np.abs(la - zs), np.abs(lb - np.conj(zs)))
            j = int(np.argmin(vals))
            if vals[j] < best[0]:
                best = (float(vals[j]), float(a), float(Ws[j]))
        ac, wc = best[1], best[2]
        half /= 8.0
    return best


tests = [
    ("2 reelles distinctes", -0.4598 + 0j, -2.5902 + 0j),
    ("reelle double",        -1.5000 + 0j, -1.5000 + 0j),
    ("paire conjuguee",      -1.1500 + 0.9932j, -1.1500 - 0.9932j),
    ("paire conj. quelconque", 0.3 + 2.7j, 0.3 - 2.7j),
    ("reelle + complexe",    -0.5936 + 0j, -1.3000 + 0.7868j),
]
print(f"{'cas':>23} | {'forme close':>12} | {'min numerique':>13} | {'ecart':>10} | {'omega* trouve':>13}")
print("-" * 84)
for name, la, lb in tests:
    closed = abs(la - np.conj(lb)) / 2.0
    val, a_star, w_star = brute_min(la, lb)
    print(f"{name:>23} | {closed:12.8f} | {val:13.8f} | {abs(val-closed):10.2e} | {w_star:13.6f}")

**Lecture.** La forme close et la minimisation numérique aveugle — qui part de $(0,0)$ et non
de la réponse — coïncident à $\sim 10^{-8}$ près. L'écart résiduel est la **résolution de la
grille** au dernier tour, pas une erreur de la formule : il décroît si l'on ajoute des tours.
Les trois lignes qui portent le théorème :

- **paire conjuguée** $\Rightarrow$ distance $0$ : le prior est *exactement* correct ;
- **réelle double** $\Rightarrow$ distance $0$, avec $\omega^\star = 0$ : le cas critique est
  dans la classe, à la frontière ;
- **deux réelles distinctes** $\Rightarrow$ distance $|\lambda_a-\lambda_b|/2 > 0$, avec
  $\omega^\star = 0$ : le bloc « fait la moyenne » des deux taux et ne peut pas mieux.

La dernière ligne (réelle + complexe) est là pour rappeler que la formule
$\frac{1}{2}|\lambda_a - \bar\lambda_b|$ est générale et ne suppose pas que la paire soit
conjuguée ou réelle. Elle servira au §3.4.

### 3.4 L'axe de mauvaise spécification $\delta(c)$

On dispose maintenant de tout : le spectre de $A(c)$ (§3.2) et la distance d'une paire à un
bloc (§3.3). Il reste à décider **comment apparier** les 4 valeurs propres de $A$ aux 2 blocs
du prior. C'est un choix de convention, il change les chiffres, et il doit donc être déclaré
avant de les regarder.

**Convention déclarée.** *Une paire complexe conjuguée n'est jamais scindée entre deux blocs.*
Autrement dit, $\delta(c)$ est le minimum, sur les appariements des 4 valeurs propres en 2
paires **qui respectent la conjugaison**, du maximum des distances par paire :

$$\delta(c) \;=\; \min_{\substack{\text{appariements}\\ \text{respectant la conjugaison}}}\;\;
\max_{\{\lambda_a,\lambda_b\}\ \in\ \text{appariement}} \;\frac{|\lambda_a - \bar\lambda_b|}{2}.$$

**Justification, et le prix de la convention.** Une paire conjuguée est représentable à coût
nul ; la scinder revient à dépenser deux blocs pour une paire déjà gratuite. La convention
garde donc $\delta$ interprétable comme *« combien de modes physiques sortent de la classe »*.
Elle n'est **pas** la borne inférieure absolue : l'appariement libre, qui autorise un bloc à
chevaucher une paire conjuguée, peut donner strictement moins. Nous calculons **les deux**
courbes et nous mesurons l'écart, plutôt que de le passer sous silence.

In [ ]:
# --- [DEMONTRE -> calcule] delta(c) sous les deux conventions ------------------------------
PAIRINGS = [((0, 1), (2, 3)), ((0, 2), (1, 3)), ((0, 3), (1, 2))]


def pair_dist(la, lb):
    """Distance minimale d'un bloc rotation-scaling a la paire {la, lb}  (theoreme 3.3)."""
    return abs(la - np.conj(lb)) / 2.0


def respects_conjugation(ev, p, q, tol=1e-9):
    """Vrai si aucune paire conjuguee n'est scindee par l'appariement (p, q)."""
    for (i, j) in (p, q):
        if abs(ev[i].imag) > tol or abs(ev[j].imag) > tol:
            if abs(ev[i] - np.conj(ev[j])) > tol:
                return False
    return True


def delta_of_c(c, free=False):
    ev = np.linalg.eigvals(A_state(c))
    best = np.inf
    for (p, q) in PAIRINGS:
        if (not free) and (not respects_conjugation(ev, p, q)):
            continue
        best = min(best, max(pair_dist(ev[p[0]], ev[p[1]]),
                             pair_dist(ev[q[0]], ev[q[1]])))
    return best


cs = np.linspace(0.01, 4.50, 1400)
d_decl = np.array([delta_of_c(c) for c in cs])
d_free = np.array([delta_of_c(c, free=True) for c in cs])

fig, ax = plt.subplots(figsize=(8.2, 4.0))
ax.plot(cs, d_decl, lw=2.0, label=r"$\delta(c)$ — convention declaree (paires conjuguees non scindees)")
ax.plot(cs, d_free, lw=1.4, ls="--", color="tab:orange", label=r"appariement libre (borne inferieure)")
for cst, lab, ytxt in zip(c_star, [r"$c^*_1=2\sqrt{\kappa_1}$", r"$c^*_2=2\sqrt{\kappa_2}$"],
                          [0.60, 1.15]):
    ax.axvline(cst, color="k", ls=":", lw=1.1)
    ax.text(cst + 0.05, ytxt, lab + f"\n{cst:.3f}", fontsize=8,
            bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))
ax.axvspan(0.0, c_star[0], color="tab:green", alpha=0.08)
ax.text(1.05, 0.10, "prior EXACT\n(2 paires conjuguees)", ha="center", fontsize=8.5, color="darkgreen")
ax.axvspan(c_star[1], 4.5, color="tab:red", alpha=0.06)
ax.text(3.95, 0.70, "4 valeurs propres reelles :\n$\\delta$ REDESCEND", ha="center", fontsize=8.5,
        color="darkred")
tab_c = [0.10, 2.30, 2.60, 3.00, 3.05, 4.00]
ax.plot(tab_c, [delta_of_c(c) for c in tab_c], "ko", ms=5, zorder=5)
ax.set(xlabel="amortissement visqueux $c$", ylabel=r"defaut spectral $\delta$",
       title=r"$\delta(c)$ : de combien le spectre de $A(c)$ sort de la classe rotation-scaling")
ax.legend(loc="upper left"); ax.set_ylim(-0.05, 1.30)
plt.tight_layout(); plt.show()

print(f"{'c':>7} | {'delta declare':>13} | {'appariement libre':>17} | {'nature du spectre':>34}")
print("-" * 82)
for c in tab_c:
    ev = np.linalg.eigvals(A_state(c))
    n_real = int(np.sum(np.abs(ev.imag) < 1e-9))
    nature = {0: "2 paires conjuguees", 2: "1 paire conj. + 2 reelles", 4: "4 reelles"}[n_real]
    print(f"{c:7.2f} | {delta_of_c(c):13.3f} | {delta_of_c(c, True):17.3f} | {nature:>34}")

mono = np.all(np.diff(d_decl[cs <= c_star[1]]) >= -1e-12)
print(f"\ndelta declare monotone croissant sur [0, c*_2] : {bool(mono)}")
print(f"delta declare juste avant c*_2 = {delta_of_c(c_star[1] - 1e-4):.3f}"
      f"  ->  juste apres = {delta_of_c(c_star[1] + 1e-2):.3f}  (discontinuite de la convention)")

**Lecture.** La courbe $\delta(c)$ se lit en trois zones.

1. **$c < c^*_1 = 2.183$ (zone verte).** Les deux modes sont sous-amortis, le spectre de $A$
   est constitué de deux paires conjuguées, et $\delta = 0$ : **le prior rotation-scaling est
   exactement correct**. À $c = 0.10$, la classe d'hypothèses contient la solution exacte.
   C'est le régime de la campagne D3 du dépôt (le contrôle D3, présenté en §3.5), et c'est ce
   qui rend son résultat interprétable — développé au notebook 04.
2. **$c^*_1 < c < c^*_2 = 3.039$.** Le premier mode devient suramorti : deux valeurs propres
   réelles distinctes, non représentables. $\delta$ croît continûment et **monotonement**
   (vérifié dans la sortie) de $0$ à $\approx 1.06$. C'est l'axe utilisable : un seul scalaire
   physique règle la mauvaise spécification.
3. **$c > c^*_2$ (zone rouge).** Le second mode bascule à son tour. Les 4 valeurs propres sont
   réelles, et le prior peut alors **réapparier** : au lieu de dépenser un bloc sur l'énorme
   écart intra-mode, il apparie les deux réelles proches entre elles. $\delta$ **redescend**
   ($0.468$ à $c = 3.05$, $0.188$ à $c = 4.00$). L'axe cesse d'être monotone, donc il cesse
   d'être un axe : deux valeurs de $c$ y correspondent au même $\delta$ avec des dynamiques
   physiques opposées. **C'est la raison pour laquelle le balayage s'arrête à $c = 3.039$**,
   et cette raison est structurelle, pas esthétique.

**La table des valeurs, arrondie au millième :** $c = 0.10 \to \delta = 0.000$ ;
$c = 2.30 \to 0.363$ ; $c = 2.60 \to 0.706$ ; $c = 3.00 \to 1.029$ ; puis, au-delà du second
seuil, $c = 3.05 \to 0.468$ et $c = 4.00 \to 0.188$.

**Le prix de la convention, mesuré.** La courbe orange (appariement libre) décroche de la
courbe déclarée entre $c^*_1$ et $c^*_2$ : à $c = 2.60$, $0.529$ contre $0.706$ ; à
$c = 3.00$, $0.529$ contre $1.029$. Autrement dit, un bloc a **le droit** de chevaucher une
paire conjuguée et d'y gagner. La convention déclarée est donc une **sur-estimation** du
défaut réel dans cette zone, d'un facteur $\approx 2.0$. On la garde parce qu'elle est
monotone et interprétable, mais on ne peut pas la présenter comme la distance minimale. La
sortie mesure aussi sa conséquence : $\delta$ déclaré est **discontinu** en $c^*_2$ (saut de
$1.057$ à $0.471$), la version libre non.

**Ce qu'on ne peut pas conclure.** (i) Que $\delta$ prédise une erreur d'apprentissage : c'est
une distance **spectrale**, et le lien entre défaut spectral et erreur de prédiction dépend
du conditionnement de la base propre — ici $\mathrm{cond}(V) = 1$, ce qui est le cas le plus
favorable, et donc le moins généralisable. (ii) Que $\delta = 0$ garantisse quoi que ce soit
sur l'entraînement. C'est exactement le sujet de la section suivante.

### 3.5 Le contrôle D3, dont ce système est le banc

Dans le régime $\delta = 0$ de §3.4 — le
prior rotation-scaling est exactement correct — le dépôt a couru une campagne où la solution
exacte est récupérable en une ligne de moindres carrés, et où le régime d'entraînement
déclaré atterrit pourtant à environ deux décades au-dessus du seuil d'exactitude
pré-enregistré, malgré quatre explications concurrentes testées et éliminées une par une.
C'est le résultat qui sépare « le prior contient la solution » de « la descente de gradient
la trouve », et qui isole le rollout fermé de 25 pas comme l'ingrédient qui suffit à créer
l'écart. La campagne, les quatre éliminations et la maquette qui isole cet ingrédient sont
développées au notebook 04 (`04_ce_que_la_grille_a_dit.ipynb`), qui prolonge directement le
système conducteur de §3.

### 3.6 La cellule structurée, et sa jauge exacte

Le bras à structure ON du dépôt lift l'état
latent par un MLP $\psi$ et le redescend par un MLP $\pi$ indépendant, sans terme de
reconstruction ni contrainte d'inversibilité entre les deux — ce qui ouvre une orbite de
paramètres de dimension $2r$ le long de laquelle la prédiction latente ne change pas d'un
bit. C'est une dégénérescence exacte, vérifiée à la précision machine, et son effet sur ce
que la descente de gradient atteint reste une question ouverte : le dépôt a retiré une
conclusion antérieure qui allait trop vite sur ce point. Le théorème de jauge, le compte des
$2r$ directions plates, et le correctif $\pi = \psi^{+}$ dans le cas linéaire sont
développés au notebook 02 (`02_les_bras_du_factoriel_et_l_equite.ipynb`), qui construit aussi
le wrapper $\psi/\bar A/\pi$ commun aux backbones du factoriel (`mlp`, `minimal causal
transformer`, `min_ssm`).

## 4. Ce que la taille d'échantillon change

### 4.1 L'argument, et pourquoi il est décisif

Un biais inductif n'achète pas de la précision : il achète de l'**efficacité d'échantillonnage**.
C'est une distinction élémentaire et elle décide de la lecture du contrôle D3 (notebook 04).

**[MESURÉ — `PROTOCOL.md §20.1`]** Le régime de données de la campagne D3 est
`n_episodes = 256`, soit **24 480 transitions**, pour un système linéaire à **4 états**. Sur ce
split, les moindres carrés atteignent une erreur spectrale de $2.11\times10^{-8}$.

Comptons ce que cela signifie. Un LTI $x_{t+1} = Ax_t + Bu_t$ avec $d = 4$ états et une action
scalaire a

$$p_{\text{libre}} \;=\; \underbrace{d^2}_{A\ :\ 16} + \underbrace{d\cdot d_u}_{B\ :\ 4} \;=\; 20
\quad\text{paramètres réels},$$

et chaque transition observée fournit $d = 4$ équations scalaires. Le seuil d'identifiabilité
est donc $4n \ge 20$, soit $n \ge 5$ transitions. Avec la contrainte rotation-scaling, $A$ n'a
plus que $2r = 4$ paramètres (deux blocs, $(\alpha_i,\omega_i)$ chacun), donc
$p_{\text{contraint}} = 4 + 4 = 8$ et le seuil tombe à $n \ge 2$.

À $n = 24\,480$, on dispose de $97\,920$ équations pour $20$ inconnues : un facteur
$\approx 4\,900$, **trois ordres de grandeur de sur-détermination**. Dans ce régime, un prior
correct ne peut rien acheter, parce qu'il n'y a rien à acheter : le problème est résolu par
comptage.

### 4.2 Pourquoi le gain doit décroître — la forme de la loi

**Hypothèses.** (H1) modèle bien spécifié, $x_{t+1} = Ax_t + w_t$, $w_t$ centré, de covariance
$\sigma^2 I$, indépendant du design ; (H2) design $X$ de $n$ lignes avec
$X^\top X \approx n\,I_d$ (états blanchis) ; (H3) estimateur des moindres carrés dans une
classe **linéaire** de $p$ paramètres.

Pour l'estimateur libre, $\hat A - A = \big(X^\top X\big)^{-1}X^\top W$ transposé, d'où

$$\mathbb{E}\,\|\hat A - A\|_F^2 \;=\; \sigma^2\, d\;\mathrm{tr}\big((X^\top X)^{-1}\big)
\;\approx\; \sigma^2\,\frac{d^2}{n} \;=\; \sigma^2\,\frac{p_{\text{libre}}^{(A)}}{n}.$$

Le même calcul dans une sous-classe linéaire de dimension $p$ (les blocs rotation-scaling en
sont une : l'application $(a,b)\mapsto \begin{pmatrix} a & -b\\ b & a\end{pmatrix}$
est **linéaire** en $(a,b)$, donc l'estimateur contraint est lui aussi un moindre carré
ordinaire) donne $\mathbb{E}\|\hat A - A\|_F^2 \approx \sigma^2 p/n$. Donc

$$\boxed{\;\|\hat A - A\|_F \;\sim\; \sigma\sqrt{\frac{p}{n}},
\qquad \frac{\text{erreur libre}}{\text{erreur contrainte}} \;\xrightarrow[n\to\infty]{}\;
\sqrt{\frac{p_{\text{libre}}^{(A)}}{p_{\text{contraint}}^{(A)}}} = \sqrt{\frac{16}{4}} = 2.\;}$$

**Trois conséquences, et la troisième est celle qu'on lit mal.**

1. Sous le seuil d'identifiabilité du modèle libre ($n < 5$), le prior n'apporte pas un gain
   de facteur 2 : il apporte la différence entre **estimable** et **non estimable**. Le gain
   y est arbitrairement grand.
2. En régime asymptotique, le **rapport** des erreurs tend vers une constante ($2$ ici), il ne
   tend pas vers $1$. Écrire « l'écart se referme » sans préciser en quel sens est faux.
3. En revanche l'écart **absolu** $\sigma(\sqrt{p_1}-\sqrt{p_2})/\sqrt{n}$ s'effondre en
   $1/\sqrt{n}$, et c'est lui qui décide face à un seuil de tâche fixé. À $n = 24\,480$, les
   deux estimateurs sont sous n'importe quel seuil raisonnable : le prior est correct, gratuit,
   et **sans effet mesurable**. C'est exactement le régime de D3.

**La prédiction pré-enregistrée du dépôt.** La campagne déclarée en `PROTOCOL.md §23`
(amendement 14, écrit avant tout run) fixe sa règle de lecture primaire **avant** les chiffres :
ce qui est lu est le **signe de l'interaction** $\text{gain(structure)} \times \log n$, jamais
un franchissement de seuil. L'analyse ci-dessus prédit ce signe : **négatif** — le gain de la
structure décroît quand $n$ croît. Si le signe mesuré était positif, l'analyse de §4.2 serait
fausse et devrait être retirée.

### 4.3 L'expérience jouet, et ce qu'on concède au prior

**[ILLUSTRATIF]** Deux estimateurs, deux systèmes, un balayage en $n$.

Le système conducteur $A(c)$ de §3 n'est pas repris tel quel dans ce qui suit : $A(c)$ n'est **normale** pour aucun $c$ testé (mesuré ci-dessous), donc aucune base orthogonale ne le met exactement sous forme de blocs rotation-scaling. C'est précisément cette exactitude que la comparaison isotrope qui suit exige, pour n'isoler que l'effet du comptage de paramètres. On construit donc un système synthétique minimal, qui porte la même distinction (spécifié / mal spécifié, au sens du théorème 3.3) sans hériter de ce défaut.

- **Estimateur libre** : moindres carrés sur les 16 coefficients de $A$, sans contrainte.
- **Estimateur contraint** : moindres carrés sur 4 coefficients — deux blocs rotation-scaling
  $\begin{pmatrix} a_i & -b_i\\ b_i & a_i\end{pmatrix}$ — **dans une base modale $V$
  qu'on lui donne**.

Ce dernier point est une **concession explicite au prior**, et il faut la nommer : dans un
modèle appris, la base est apprise par $\psi$ et $\pi$, elle n'est pas donnée. En la donnant,
on retire au prior son coût principal et on isole l'effet du comptage de paramètres. Le
résultat est donc une **borne optimiste** sur ce qu'un prior structurel peut acheter.
On choisit de plus $V$ **orthogonale** ($\mathrm{cond}(V) = 1$), pour que le design reste
isotrope dans la base modale — sinon le biais de mauvaise spécification dépendrait aussi de
la covariance, et on mélangerait deux effets.

Système **A** (bien spécifié) : $V\,\mathrm{blockdiag}\big(0.95R(0.7),\,0.88R(-1.9)\big)V^\top$
— le prior est exact, $\delta = 0$.
Système **B** (mal spécifié) : le premier bloc est remplacé par
$\mathrm{diag}(\lambda_a,\lambda_b)$ avec $\lambda_a = 0.95$, $\lambda_b = 0.40$, deux réelles
distinctes. Par le théorème 3.3, $\delta = |\lambda_a-\lambda_b|/2 = 0.275$.

In [ ]:
# --- [DEMONTRE -> verifie] Pourquoi A(c) n'est pas repris tel quel : non-normalite --------
print("A(c) est-elle normale (A A^T = A^T A) ? condition necessaire pour qu'une base")
print("ORTHOGONALE la mette exactement sous forme de blocs rotation-scaling :")
for _c in [0.10, 3.00]:
    _Ac = A_state(_c)
    _resid = np.abs(_Ac @ _Ac.T - _Ac.T @ _Ac).max()
    print(f"  c={_c:.2f} : |A A^T - A^T A|_inf = {_resid:.4f}  (non nul : PAS normale)")
print()

# --- [ILLUSTRATIF] Efficacite d'echantillonnage : prior exact vs prior faux -----------------
rng6 = np.random.default_rng(2024)
D6 = 4


def blkdiag2(B1, B2):
    M = np.zeros((4, 4)); M[:2, :2] = B1; M[2:, 2:] = B2
    return M


V6, _ = np.linalg.qr(rng6.standard_normal((D6, D6)))      # base modale ORTHOGONALE, donnee au prior
M_ok = blkdiag2(0.95 * rot(0.7), 0.88 * rot(-1.9))        # 2 paires conjuguees : delta = 0
LA, LB = 0.95, 0.40
M_bad = blkdiag2(np.diag([LA, LB]), 0.88 * rot(-1.9))     # 2 reelles distinctes : delta > 0
A_ok, A_bad = V6 @ M_ok @ V6.T, V6 @ M_bad @ V6.T
SIGMA = 0.05
delta_bad = abs(LA - LB) / 2

print(f"systeme A : spectre {np.round(canon_sort(np.linalg.eigvals(A_ok)), 4)}")
print(f"            delta = {0.0:.4f}  (2 paires conjuguees : prior EXACT)")
print(f"systeme B : spectre {np.round(canon_sort(np.linalg.eigvals(A_bad)), 4)}")
print(f"            delta = {delta_bad:.4f}  (theoreme 3.3 : |la - lb| / 2)")


def fit_free(Xs, Ys):
    """Moindres carres sur les 16 coefficients de A."""
    return np.linalg.lstsq(Xs, Ys, rcond=None)[0].T


def fit_constrained(Xs, Ys, V):
    """Moindres carres sur 4 coefficients : deux blocs [[a,-b],[b,a]] dans la base modale V.

    La parametrisation (a, b) -> [[a,-b],[b,a]] est LINEAIRE, donc c'est encore un moindre
    carre ordinaire -- pas une optimisation non convexe. C'est ce qui rend la comparaison
    propre : deux estimateurs de meme nature, seul p change.
    """
    Xm, Ym = Xs @ V, Ys @ V                        # V orthogonale : V^{-1} = V^T
    M = np.zeros((4, 4))
    for b in range(2):
        i = 2 * b
        x1, x2 = Xm[:, i], Xm[:, i + 1]
        y1, y2 = Ym[:, i], Ym[:, i + 1]
        Dsg = np.stack([np.concatenate([x1, x2]), np.concatenate([-x2, x1])], axis=1)
        a_b = np.linalg.lstsq(Dsg, np.concatenate([y1, y2]), rcond=None)[0]
        M[i:i + 2, i:i + 2] = np.array([[a_b[0], -a_b[1]], [a_b[1], a_b[0]]])
    return V @ M @ V.T


def sweep(A_true_, ns, reps=24):
    out = {"free": [], "cons": [], "free_q": [], "cons_q": []}
    for n in ns:
        ef, ec = [], []
        for rep in range(reps):
            rr = np.random.default_rng(97_000 + 131 * rep + n)
            Xs = rr.standard_normal((n, D6))
            Ys = Xs @ A_true_.T + SIGMA * rr.standard_normal((n, D6))
            ef.append(np.linalg.norm(fit_free(Xs, Ys) - A_true_))
            ec.append(np.linalg.norm(fit_constrained(Xs, Ys, V6) - A_true_))
        out["free"].append(np.median(ef)); out["cons"].append(np.median(ec))
        out["free_q"].append((np.quantile(ef, .25), np.quantile(ef, .75)))
        out["cons_q"].append((np.quantile(ec, .25), np.quantile(ec, .75)))
    return {k: np.array(v) for k, v in out.items()}


NS = np.array([1, 2, 3, 4, 5, 7, 10, 15, 25, 50, 100, 250, 600, 1500, 4000, 10000])
res_ok, res_bad = sweep(A_ok, NS), sweep(A_bad, NS)
print("\nbalayage termine :", len(NS), "valeurs de n, 24 repetitions chacune")

In [ ]:
# --- [ILLUSTRATIF] La figure : erreur contre n, et le plafond de biais ---------------------
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))

for ax, res, A_t, titre, exact in [
    (axes[0], res_ok, A_ok, "(a) systeme A : prior EXACT ($\\delta = 0$)", True),
    (axes[1], res_bad, A_bad, f"(b) systeme B : prior FAUX ($\\delta = {delta_bad:.3f}$)", False),
]:
    for key, lab, col in [("free", "libre (16 parametres)", "tab:blue"),
                          ("cons", "contraint (4 parametres)", "tab:red")]:
        ax.loglog(NS, res[key], "o-", ms=4, color=col, label=lab)
        ax.fill_between(NS, res[key + "_q"][:, 0], res[key + "_q"][:, 1], color=col, alpha=0.15)
    # lois theoriques sigma sqrt(p / n), sans constante ajustee
    ax.loglog(NS, SIGMA * np.sqrt(16 / NS), "--", color="tab:blue", lw=1, alpha=0.8,
              label=r"$\sigma\sqrt{16/n}$")
    ax.loglog(NS, SIGMA * np.sqrt(4 / NS), "--", color="tab:red", lw=1, alpha=0.8,
              label=r"$\sigma\sqrt{4/n}$")
    ax.axvline(5, color="k", ls=":", lw=1)
    ax.text(5.6, 4.0, "seuil d'identifiabilite\ndu modele libre ($n=5$)", fontsize=7.5,
            bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))
    ax.set_ylim(3e-4, 12.0)
    if not exact:
        ax.axhline(np.sqrt(2) * delta_bad, color="tab:red", ls="-.", lw=1.2)
        ax.text(1.6, np.sqrt(2) * delta_bad * 0.33,
                r"plafond de biais $\sqrt{2}\,\delta$", fontsize=8, color="darkred")
    ax.set(xlabel="$n$ (transitions)", ylabel=r"$\|\hat A - A\|_F$ (mediane, 24 rep.)", title=titre)
    ax.legend(fontsize=7.5, loc="lower left")

ax = axes[2]
ax.semilogx(NS, res_ok["free"] / res_ok["cons"], "o-", ms=4, color="tab:green",
            label="systeme A (prior exact)")
ax.semilogx(NS, res_bad["free"] / res_bad["cons"], "s-", ms=4, color="tab:purple",
            label="systeme B (prior faux)")
ax.axhline(1.0, color="k", lw=1)
ax.axhline(2.0, color="tab:green", ls="--", lw=1, label=r"asymptote $\sqrt{p_1/p_2}=2$")
ax.set(xlabel="$n$ (transitions)", ylabel="erreur libre / erreur contrainte",
       title="(c) gain du prior ; $>1$ = le prior aide", yscale="log")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

print(f"{'n':>7} | {'A: libre':>10} {'A: contr.':>10} {'gain':>7} | "
      f"{'B: libre':>10} {'B: contr.':>10} {'gain':>7}")
print("-" * 76)
for i, n in enumerate(NS):
    print(f"{n:7d} | {res_ok['free'][i]:10.3e} {res_ok['cons'][i]:10.3e} "
          f"{res_ok['free'][i]/res_ok['cons'][i]:7.2f} | "
          f"{res_bad['free'][i]:10.3e} {res_bad['cons'][i]:10.3e} "
          f"{res_bad['free'][i]/res_bad['cons'][i]:7.2f}")

print(f"\nplafond mesure du contraint sur le systeme B (n = {NS[-1]}) : {res_bad['cons'][-1]:.6f}")
print(f"prediction analytique sqrt(2) * delta                      : {np.sqrt(2)*delta_bad:.6f}")
print(f"ecart relatif                                              : "
      f"{abs(res_bad['cons'][-1] - np.sqrt(2)*delta_bad)/(np.sqrt(2)*delta_bad):.2e}")
cross = NS[np.argmax(res_bad["free"] / res_bad["cons"] < 1.0)]
print(f"croisement sur le systeme B (le libre repasse devant)       : n = {cross}")

**Lecture.** Trois régimes, et ils sont tous visibles.

**Panneau (a) — prior exact.** Sous $n = 5$, l'estimateur libre est **sous-déterminé** : les
moindres carrés rendent la solution de norme minimale, et l'erreur est d'ordre 1. L'estimateur
contraint, lui, est déjà identifié à $n = 2$ et se trompe dix à seize fois moins. C'est le
vrai gain d'un prior : il déplace le seuil d'identifiabilité, pas la constante. Passé
$n = 10$, les deux courbes deviennent parallèles en log-log, de pente $-1/2$, et se collent aux
lois $\sigma\sqrt{p/n}$ tracées **sans aucune constante ajustée**. Le rapport se stabilise
autour de $2 = \sqrt{16/4}$, exactement la prédiction de §4.2.

**Panneau (b) — prior faux.** Le contraint part devant (il reste identifiable là où le libre ne
l'est pas), puis **plafonne**. Le plafond est mesuré à $0.38896$, contre une prédiction
analytique $\sqrt{2}\,\delta = 0.38891$ : $1.4\times10^{-4}$ d'écart relatif. La prédiction se
dérive en une ligne : sous design isotrope dans la base modale, le moindre carré contraint
réalise la **projection orthogonale de Frobenius** de $\mathrm{diag}(\lambda_a,\lambda_b)$ sur
le sous-espace $\{aI + bJ\}$ ; or
$\mathrm{diag}(\lambda_a,\lambda_b) = \frac{\lambda_a+\lambda_b}{2}I + \frac{\lambda_a-\lambda_b}{2}\mathrm{diag}(1,-1)$
et $\mathrm{diag}(1,-1)$ est orthogonale à la fois à $I$ et à $J$. Le résidu est donc
exactement $\frac{\lambda_a-\lambda_b}{2}\mathrm{diag}(1,-1)$, de norme de Frobenius
$\frac{|\lambda_a-\lambda_b|}{2}\sqrt{2} = \sqrt{2}\,\delta$. **Le défaut spectral $\delta$ de
§3 n'est donc pas une abstraction : c'est le plancher d'erreur, à un facteur $\sqrt2$ près
qui vient de la norme choisie.**

**Panneau (c) — le croisement.** Le gain du prior est $>1$ à petit $n$ dans **les deux**
systèmes, y compris quand le prior est faux : sous le seuil d'identifiabilité, un mauvais
modèle identifiable bat un bon modèle non identifiable. Puis les deux courbes divergent :
gain $\to 2$ quand le prior est exact, gain $\to 0$ quand il est faux, avec un croisement de
la ligne $1$ à $n = 5$ — c'est-à-dire **exactement** au seuil d'identifiabilité du modèle
libre. Un prior faux n'est pas neutre : il devient nuisible dès que les données suffisent à
s'en passer.

**Ce qu'on ne peut pas conclure, et il y a beaucoup à dire.**

1. **La base modale a été donnée au prior.** Dans un modèle appris, $\psi$ et $\pi$ doivent
   la trouver, et le notebook 02 montre qu'ils ne sont contraints par rien pour le faire. Le gain
   mesuré ici est donc une **borne supérieure optimiste**.
2. **Les deux estimateurs sont des moindres carrés**, donc convexes et résolus en forme close.
   Le notebook 04 mesure que le régime d'entraînement réel n'atteint pas cet optimum. Le gain
   d'échantillonnage démontré ici est un gain **statistique**, pas un gain **atteignable** :
   les deux se composent, et rien ne dit que la composition soit favorable.
3. **Le design est i.i.d. gaussien**, pas une trajectoire. Sur une trajectoire, les états sont
   corrélés, le nombre effectif d'échantillons est plus petit que $n$, et le seuil
   d'identifiabilité se déplace vers la droite. La figure est donc optimiste sur l'axe $n$
   aussi.
4. **Le rapport ne dit rien de l'utilité.** À $n = 10\,000$, le prior exact divise l'erreur
   par 2 — mais les deux valent $\sim 10^{-3}$, et aucune tâche en aval ne distingue
   $10^{-3}$ de $2\times10^{-3}$. C'est l'argument central de §4.1, ici sous forme graphique.

### 4.4 Synthèse : ce que l'axe manquant aurait mesuré

En superposant le résultat du contrôle D3 (notebook 04) et l'analyse qui précède, la
lecture du dossier change de forme.

| | régime $n$ petit | régime $n$ grand (D3 : 24 480) |
|---|---|---|
| prior exact ($\delta = 0$) | gain d'identifiabilité, potentiellement grand | gain $\to$ facteur constant, **sans effet sur une tâche** |
| prior faux ($\delta > 0$) | gain possible malgré tout | **nuisible**, plafond $\propto \delta$ |

La campagne D3 s'est placée dans la colonne de droite, ligne du haut : le seul régime où un
prior correct ne peut **rien** démontrer. La planification initiale du projet appelait ce réglage « le meilleur
cas possible » ; l'analyse ci-dessus dit que c'est le **pire cas possible** pour faire
apparaître un biais inductif. Ce n'est pas une erreur de mesure — les 10 runs sont propres et
leur verdict tient. C'est une erreur de **placement**, et elle est réparable : c'est
exactement ce que déclare l'amendement 14 (`PROTOCOL.md §23`), en ajoutant l'axe
`n_episodes` $\in \{8, 32, 256\}$ à backbone gelé.

**Ce que ce notebook ne peut pas faire.** Prédire le résultat de cette campagne. L'analyse de
§4.2 est valable pour des **estimateurs des moindres carrés** ; le dépôt entraînera des
réseaux par descente de gradient sur un rollout fermé, régime dont le notebook 04 mesure qu'il n'atteint
pas l'optimum des moindres carrés. Le signe de l'interaction reste donc une **prédiction
falsifiable**, et c'est bien ainsi qu'elle est enregistrée.

### 4.5 L'inventaire des limites du dépôt, tenu à jour ailleurs

Les réserves propres à ce qui
reste dans ce notebook sont notées localement, cellule par cellule : le prix de la
convention d'appariement (§3.4), le statut cité, non recalculé, du taux de composition
(§1.5), le caractère prédictif et non couru de l'axe taille d'échantillon (§4.4).
L'inventaire des limites du dépôt dans son ensemble — le statut du contraste ON/OFF, tenu
sur `mlp` et `minimal causal transformer` alors que `min_ssm` reste non informatif pour une
raison pré-déclarée (`PROTOCOL.md` §27.9), l'effet non mesuré de la dégénérescence de jauge,
le régime d'entraînement en rollout fermé sans équivalent publié — devient le ledger tenu à
jour au
notebook 04 (`04_ce_que_la_grille_a_dit.ipynb`), pour n'avoir qu'une seule adresse plutôt que
de dériver d'un chapitre à l'autre.

## 5. Récapitulatif des vérifications numériques de ce notebook

| § | énoncé | vérifié contre | résultat |
|---|---|---|---|
| 1.3 | $\|e_H\| \le \varepsilon(\eta^H-1)/(\eta-1)$, atteinte au pire cas | simulation directe | superposition exacte |
| 2.2 | $\exp(\Delta G) = e^{\Delta\alpha}R(\Delta\omega)$ | série de Taylor | $\le 3\times10^{-16}$ |
| 2.3 | $r = \rho(\bar A) = \|\bar A\|_2 = e^{\Delta\alpha}$ | valeurs propres, SVD | égalité machine |
| 2.4 | forme close de $\Phi$ | quadrature fine | $\le 5.8\times10^{-13}$ (limite de la quadrature) |
| 2.5 | singularité effaçable, cancellation réelle | série vs forme brute | $7\times10^{-4}$ à $\|g\|=10^{-12}$ |
| 3.2 | $\lambda^2 + c\lambda + \kappa = 0$ | spectre de $A$ $4\times4$ | $\le 3\times10^{-15}$ |
| 3.3 | distance minimale $= \frac12\|\lambda_a - \bar\lambda_b\|$ | minimisation aveugle depuis $(0,0)$ | $\sim 10^{-8}$ (pas de grille) |
| 4.3 | plancher $= \sqrt{2}\,\delta$ pour un prior faux | balayage en $n$, 24 répétitions | $1.4\times10^{-4}$ relatif |

## 6. Où aller ensuite

### 6.1 Ce qui a été prouvé ici

Ce notebook a posé la récurrence de l'erreur et sa borne géométrique (§1), dérivé en forme
close l'algèbre du bloc rotation-scaling et son intégrale ZOH — avec le piège de cancellation
numérique et son correctif (§2) — construit un système où le défaut de spécification $\delta$
du prior se règle par un seul scalaire physique et démontré le théorème de représentabilité
qui le fonde (§3), puis établi pourquoi le gain d'un prior correct s'effondre, et celui d'un
prior faux se plafonne, avec la taille d'échantillon (§4). Deux autres faits fondateurs, cités
en §0.1, sont prouvés ailleurs : l'écart entre exprimable et atteignable au notebook 04, la
dégénérescence de jauge au notebook 02.

### 6.2 La carte de la série

| # | notebook | ce qu'il apporte |
|---|---|---|
| 00 | celui-ci | le problème et l'instrument |
| 01 | `01_du_neurone_a_l_operateur.ipynb` | d'où viennent les blocs $2\times 2$ (du neurone adLIF), et pourquoi le rayon doit être borné |
| 02 | `02_les_bras_du_factoriel_et_l_equite.ipynb` | les backbones, le wrapper $\psi/\bar A/\pi$, le théorème de jauge, le budget $\pm5\%$ |
| 03 | `03_entrainer_et_mesurer.ipynb` | la loss JEPA/EMA/stop-gradient, $e_H$, VPT, les gardes |
| 04 | `04_ce_que_la_grille_a_dit.ipynb` | l'identifiabilité, les campagnes, le factoriel primaire, le ledger |
| 05 | `05_l_experience_oracle.ipynb` | l'expérience oracle §28 et sa figure |
| 06 | `06_figures.ipynb` | les figures régénérées depuis les artefacts |

### 6.3 Lien avec `sld`, la limite honnête, et le pont vers le 01

| module | ce qu'il implémente |
|---|---|
| `src/sld/predictors/blocks2x2.py` | le bloc rotation-scaling du §2 |
| `src/sld/systems/representability.py` | le spectre de $A(c)$ et $\delta(c)$ du §3 |
| `src/sld/systems/coupled_oscillators.py` | le système conducteur (oscillateurs couplés), et le contrôle D3 dont le notebook 04 mesure l'exactitude |
| `src/sld/experiments/` | les runs et les gates (phase D, D3, D4, la racine factorielle) |
| `src/sld/viz/` | les figures régénérées par `06_figures.ipynb` |

Ce notebook reste **numpy-pur** par confort de lecture — il s'exécute sans installer le
dépôt — pas par principe. La parité package de niveau 3 (`REGLES.md §8`), la cellule qui
confronte la version pédagogique à la version `sld` sur le même exemple, vit dans les
notebooks qui reflètent effectivement le code entraîné : 02, 03 et 05.

La limite honnête de ce chapitre : tout ce qui y est démontré l'est sur un système linéaire
jouet, ou par simulation illustrative explicitement marquée comme telle. Aucun chiffre d'ici
n'est un verdict d'entraînabilité sur un backbone réel — ce verdict, avec ses confondants et
sa lecture pré-enregistrée, vit au notebook 04.

Le 00 a montré **que** la structure coûte, et ce qu'elle achète : une composition d'erreur
plus lente dans 3 cas sur 4 (§1.5), une lecture exacte du rayon spectral (§2.3), un axe de
mauvaise spécification mesurable (§3.4), et un gain d'échantillonnage qui s'effondre
exactement au régime où le dépôt a mesuré (§4). Le 01 montre **d'où** viennent les blocs
$2\times 2$ eux-mêmes — du neurone adLIF au sous-seuil linéarisé — et pourquoi le rayon
spectral, ici laissé libre (§2.3), doit être borné.

## Références

- **`PROTOCOL.md`** du dépôt `structured-latent-dynamics` : §13.2 (rayon libre), §20.1
  (baseline moindres carrés sur le régime de données de la campagne D3), §21.0 (table
  $e_1$/$e_{25}$), §21.2 (asymétrie de taux), §23 (amendement 14, campagne taille
  d'échantillon).